**Restart the Jupyter kernel before running any new session.**

Kernel > Restart Kernel and Clear All Outputs


# The Creative Prism
## Phase 2 — Full Studio Workflow
**v4.0 — Hybrid Model Architecture**

```
Stage 0    Routing              ← Director classifies: STUDIO / DIRECT / OUT_OF_SCOPE
Stage 1    Adaptive Discovery   ← Director-controlled loop, 1-6 turns, extraction toolkit
Stage 2    Reframing            ← Director proposes reframing, PIL confirms (branch: A/B/C)
Stage 3    Creative Brief       ← JSON format, validation gate, pressure point required
Stage 3a   Team Configuration   ← Director tunes Creative Team traits for this problem
Stage 3b   Assumption Validation← Director states premises as numbered list, PIL corrects
Stage 4a   Creator Pass 1       ← 3 directions, banded Verbalized Sampling, Step 0 naming
Stage 4b   Critic Pass 1        ← cross-model evaluation, Surprise Audit, RESEARCH_REQUEST
Stage 4c   Researcher           ← responds to requests + autonomous contribution
Stage 4d   Creator Pass 2       ← refines directions based on research
Stage 4e   Critic Pass 2        ← cross-model final evaluation + synthesis → Director
Stage 5    Candidate Directions ← Director selects 3, JSON format, validation gate
Stage 6    Director Review      ← PRESENT or ITERATE decision
Stage 6a   Iteration Loop       ← re-runs full Creative Team if ITERATE
Stage 7    Presentation         ← five-part format, full Research Foundation
Stage 8    User Reaction        ← Director reads signals, honors explicit requests
Stage 8b   Depth Extraction     ← one extraction technique, minimum signal threshold
Stage 8c   Focused Refinement   ← Director restatement → Creator → PIL mini-presentation
Stage 9    Final Synthesis      ← research-anchored, phased action plan
Stage 9b   Research Appendix    ← full research trace surfaced for PIL review
```

**v4.0 — Hybrid Model Architecture**
- Cell 2b: Model Routing — classifies problem orientation (Creative / Strategic), selects primary and Critic providers automatically
- Critic (Cells 6, 7c) runs on opposite provider from primary — Claude Critic challenges for constraint and viability; GPT Critic challenges for distinctiveness and creative surprise
- engine_hybrid.py — unified engine holding both Anthropic and OpenAI clients
- prompts_hybrid/ — shared prompt set plus critic_gpt.md for GPT Critic orientation

**v3.2**
- Presentation Core Move expanded to 8-12 sentences; Research Foundation written for PIL who has not seen the research
- Stage 9 synthesis research-anchored with phased action plan; Stage 9b Research Appendix added

**v3.1**
- Stage 8c Focused Refinement; Stage 8b minimum signal threshold; Stage 8→9 gap closed; Transparency on request

---
## Cell 1 — Load Phase 0 Foundation

In [1]:
# ── CELL 1 — LOAD FOUNDATION (Hybrid) ────────────────────────────────────────
# Imports from engine_hybrid.py.
# All other cells are identical to v3_1 except:
#   Cell 2  — Session Configuration (sets provider-aware model constants)
#   Cell 2b — Model Routing (new — runs classifier, sets routing vars)
#   Cell 6  — Critic Pass 1 (passes provider=CRITIC_PROVIDER, model=CRITIC_MODEL)
#   Cell 7c — Critic Pass 2 (same change)

import re
import json

from engine_hybrid import (
    create_blackboard,
    scribe_log,
    build_prompt,
    call_role,
    save_session,
    verify_engine,
    create_brief_doc,
    update_brief_doc,
    read_brief_doc,
    load_traits_matrix,
    weight_to_band,
    build_trait_profile,
    get_tunable_traits,
    validate_adjustments,
    validate_stage,
    generate_baseline,
    assemble_evaluation_payload,
    route_session,
    ANTHROPIC_DEFAULT,
    ANTHROPIC_DIRECTOR,
    OPENAI_DEFAULT,
    OPENAI_DIRECTOR,
)

engine_ready = verify_engine()
if not engine_ready:
    raise RuntimeError(
        "Engine verification failed.\n"
        "Setup:\n"
        "  mkdir prompts_hybrid && cp prompts/* prompts_hybrid/\n"
        "  mkdir sessions_hybrid\n"
        "  Create prompts_hybrid/critic_gpt.md\n"
        "  Ensure ANTHROPIC_API_KEY and OPENAI_API_KEY are in .env"
    )

traits_matrix = load_traits_matrix()
print(f"\nTraits matrix: {len(traits_matrix)} active traits")
print("Hybrid engine ready")

-- ENGINE VERIFICATION (Hybrid) ----------------------------
  Anthropic client: key ...MgAA
  OpenAI client:    key ...hV4A

  OK  prompts_hybrid/SYSTEM_PROMPT.md <- constitutional layer
  OK  prompts_hybrid/director.md
  OK  prompts_hybrid/creator.md
  OK  prompts_hybrid/critic.md
  OK  prompts_hybrid/critic_gpt.md <- GPT critic
  OK  prompts_hybrid/researcher.md
  OK  prompts_hybrid/scribe.md
  OK  prompts_hybrid/director_extraction_games.md

  OK  persona_traits_matrix_v2.csv (57 active traits)

  System foundation: ~1085 tokens
  All checks passed — hybrid engine ready
------------------------------------------------------------

Traits matrix: 57 active traits
Hybrid engine ready


---
## Cell 2 — Session Configuration

All PIL interactions are live input.

In [2]:
# ── CELL 2 — SESSION CONFIGURATION (Hybrid) ──────────────────────────────────
# Model constants are placeholders here — overwritten by Cell 2b after routing.
# After Cell 2b runs, PRIMARY_PROVIDER and CRITIC_PROVIDER are pushed into
# engine_hybrid so every call_role() in the session uses the correct provider
# automatically without needing provider= on every call.

import engine_hybrid

# Placeholders — overwritten by Cell 2b
DIRECTOR_MODEL = ANTHROPIC_DIRECTOR  # Sonnet — PIL-facing calls
DIRECTOR_FAST_MODEL = ANTHROPIC_DEFAULT  # Haiku  — internal/admin calls
SESSION_MODEL = ANTHROPIC_DEFAULT
CRITIC_MODEL = OPENAI_DEFAULT
PRIMARY_PROVIDER = "anthropic"
CRITIC_PROVIDER = "openai"

# Session-level trait adjustments — populated by Director in Stage 3a
session_adjustments = {"creator": {}, "critic": {}, "researcher": {}}

# Built trait profiles — populated after Stage 3a
trait_profiles = {"creator": "", "critic": "", "researcher": ""}

# Push provider state into engine so all call_role() calls inherit it.
# Cell 2b will overwrite these after the routing decision.
engine_hybrid.PRIMARY_PROVIDER = PRIMARY_PROVIDER
engine_hybrid.CRITIC_PROVIDER = CRITIC_PROVIDER

print("Session configuration ready.")
print("Run Cell 2b to classify the problem and set model routing.")

Session configuration ready.
Run Cell 2b to classify the problem and set model routing.


---
## Cell 2b — Model Routing

Classifies the problem orientation and reads the person in the loop (PIL).
Run after Cell 2 and before Cell 3.

**Claude (Sonnet) is always primary.** GPT is always Critic.
Orientation scoring determines discovery and synthesis emphasis — not model selection.

**Strategic orientation** (score ≤ 0.35) → grounding-first, operational scaffolding
**Creative orientation** (score ≥ 0.65) → expansion-first, identity and positioning
**Balanced** (0.36–0.64) → tiebreaker questions decide emphasis

PIL reading dimensions (from initial prompt language):
- Domain fluency: high / developing / low
- Cognitive style: analytical / intuitive / mixed
- Relational proximity: personal / professional / mixed

Low domain fluency or high personal proximity biases score toward strategic.
The operator can override model constants before proceeding.

In [3]:
# ── CELL 2b — MODEL ROUTING ───────────────────────────────────────────────────
# Captures the initial prompt, classifies orientation and PIL profile,
# and sets model routing based on orientation:
#
#   Creative  (score ≥ 0.65) → GPT primary, Claude Critic, Claude Scribe
#   Strategic (score ≤ 0.35) → Claude primary, GPT Critic, Claude Scribe
#   Balanced  (0.36–0.64)    → tiebreaker decides
#
# Model assignment per orientation:
#   Creative:  Director=GPT-5.4, FastModel=GPT-mini, Team=GPT-mini,
#              Critic=Claude-Haiku, Scribe=Claude-Haiku
#   Strategic: Director=Claude-Sonnet, FastModel=Claude-Haiku, Team=Claude-Haiku,
#              Critic=GPT-mini, Scribe=Claude-Haiku
#
# Scribe is always Anthropic (Claude-Haiku) regardless of orientation.

print("═" * 60)
print("MODEL ROUTING")
print("═" * 60)

# ── Capture initial prompt — used by routing AND passed to Cell 3 ─────────────
initial_prompt = input(
    "Welcome to The Creative Prism. What would you like to explore today?\n> "
).strip()

if not initial_prompt:
    print("⚠ No prompt entered. Defaulting to Claude primary (strategic).")
    _routing = {
        "orientation": "strategic",
        "orientation_score": 0.5,
        "primary_provider": "anthropic",
        "critic_provider": "openai",
        "director_model": ANTHROPIC_DIRECTOR,
        "session_model": ANTHROPIC_DEFAULT,
        "critic_model": OPENAI_DEFAULT,
        "tiebreaker_used": False,
        "tiebreaker_answers": [],
        "rationale": "No prompt provided — defaulted to strategic.",
        "pil_read": "",
        "domain_fluency": "",
        "cognitive_style": "",
        "relational_proximity": "",
    }
else:
    print("\n  Classifying...")
    _routing = route_session(initial_prompt)

# ── Set model constants from routing decision ─────────────────────────────────
DIRECTOR_MODEL = _routing["director_model"]  # Sonnet-tier: Anthropic or GPT
SESSION_MODEL = _routing["session_model"]  # Haiku-tier: Anthropic or GPT-mini
CRITIC_MODEL = _routing["critic_model"]  # Always opposite provider
PRIMARY_PROVIDER = _routing["primary_provider"]  # "anthropic" or "openai"
CRITIC_PROVIDER = _routing["critic_provider"]  # always opposite of primary

# DIRECTOR_FAST_MODEL follows the primary provider —
# internal Director calls use the fast model of whichever provider is primary.
# Scribe is always Claude Haiku regardless of orientation.
DIRECTOR_FAST_MODEL = (
    OPENAI_DEFAULT if PRIMARY_PROVIDER == "openai" else ANTHROPIC_DEFAULT
)
SCRIBE_MODEL = ANTHROPIC_DEFAULT  # always Anthropic

# ── Push provider state into engine_hybrid ───────────────────────────────────
engine_hybrid.PRIMARY_PROVIDER = PRIMARY_PROVIDER
engine_hybrid.CRITIC_PROVIDER = CRITIC_PROVIDER

# ── Print routing decision ────────────────────────────────────────────────────
print()
print(
    f"  Orientation    : {_routing['orientation'].upper()}  "
    f"(score: {_routing['orientation_score']:.2f})"
)
print(f"  Rationale      : {_routing['rationale']}")
print()
print(f"  PIL read       : {_routing.get('pil_read', '')}")
print(f"  Domain fluency : {_routing.get('domain_fluency', '')}")
print(f"  Cognitive style: {_routing.get('cognitive_style', '')}")
print(f"  Relational     : {_routing.get('relational_proximity', '')}")
if _routing["tiebreaker_used"]:
    print(f"\n  Tiebreaker     : used")
    for a in _routing["tiebreaker_answers"]:
        print(f"    {a}")
print()
print(f"  Director model : {DIRECTOR_MODEL}  [{PRIMARY_PROVIDER}]")
print(f"  Fast model     : {DIRECTOR_FAST_MODEL}  [{PRIMARY_PROVIDER}]")
print(f"  Team model     : {SESSION_MODEL}  [{PRIMARY_PROVIDER}]")
print(f"  Critic model   : {CRITIC_MODEL}  [{CRITIC_PROVIDER}]")
print(f"  Scribe model   : {SCRIBE_MODEL}  [anthropic]")
print("═" * 60)
print()
print("✓ Routing complete — proceed to Cell 3")

════════════════════════════════════════════════════════════
MODEL ROUTING
════════════════════════════════════════════════════════════

  Classifying...

  Orientation    : CREATIVE  (score: 0.62)
  Rationale      : The problem requires creative/existential thinking about identity, meaning, and what constitutes a fulfilling life, despite surface-level financial considerations being present.

  PIL read       : Someone caught between competing values (emotion vs. practicality) who recognizes both dimensions but lacks a framework to integrate them, and needs permission to weight intangible factors alongside financial ones.
  Domain fluency : developing
  Cognitive style: mixed
  Relational     : personal

  Tiebreaker     : used
    Q1: creative — Person needs alignment with what feels meaningful, not just executable steps
    Q2: creative — Bigger risk is choosing wrong emotionally and living inauthentically, not financial failure
    Q3: creative — No formula exists; requires reframin

---
## Cell 3 — Stage 0: Routing


In [4]:
# ── CELL 3 — STAGE 0: ROUTING ─────────────────────────────────────────────────
# initial_prompt is captured in Cell 2b — do not add input() here.

# -- Scenario metadata (operator setup — edit before running) -----------------
# For experimental runs: set SCENARIO_ID and SCENARIO_CATEGORY before running.
# For live sessions: leave both as empty strings. PIL never sees this.
_scenario_id = ""  # e.g. "C-01"
_scenario_category = ""  # e.g. "conventional"
if _scenario_id:
    print(f"  Scenario: {_scenario_id} [{_scenario_category}]")

print("═" * 60)
print("STAGE 0 — ROUTING")
print("═" * 60)

# initial_prompt already set in Cell 2b
print(f"  Prompt: {initial_prompt[:80]}{'...' if len(initial_prompt) > 80 else ''}")

# -- Baseline generation (Control B) ------------------------------------------
baseline_response = generate_baseline(
    initial_prompt,
    model=SESSION_MODEL,
    provider=PRIMARY_PROVIDER,
)
print("  Baseline generated and stored.")

# -- Create session ------------------------------------------------------------
blackboard = create_blackboard(
    initial_prompt,
    system_version="hybrid-1.0",
    director_model=DIRECTOR_MODEL,
    primary_provider=PRIMARY_PROVIDER,
)
session_id = blackboard["session_metadata"]["session_id"]
brief_doc_path = create_brief_doc(session_id, initial_prompt)
scribe_log(
    blackboard, "SYSTEM", "session_start", f"Session initiated: '{initial_prompt}'"
)

# -- Store routing decision in blackboard -------------------------------------

blackboard["routing"].update(
    {
        "orientation": _routing["orientation"],
        "orientation_score": _routing["orientation_score"],
        "primary_provider": PRIMARY_PROVIDER,
        "critic_provider": CRITIC_PROVIDER,
        "tiebreaker_used": _routing["tiebreaker_used"],
        "tiebreaker_answers": _routing["tiebreaker_answers"],
        "rationale": _routing["rationale"],
        "pil_read": _routing.get("pil_read", ""),
        "domain_fluency": _routing.get("domain_fluency", ""),
        "cognitive_style": _routing.get("cognitive_style", ""),
        "relational_proximity": _routing.get("relational_proximity", ""),
    }
)


# -- Write scenario metadata to blackboard (silent) ---------------------------
if _scenario_id:
    blackboard["session_metadata"]["scenario_metadata"] = {
        "scenario_id": _scenario_id,
        "category": _scenario_category,
        "prompt_as_entered": initial_prompt,
    }

# -- Director routing classification call -------------------------------------
# Returns STUDIO / DIRECT / OUT_OF_SCOPE — not shown to PIL
routing_task = (
    "A person has just arrived at The Creative Prism with this request:\n\n"
    f'"{initial_prompt}"\n\n'
    "Determine the right route for this request.\n\n"
    "STUDIO — A creative challenge that benefits from structured exploration: "
    "brand development, strategic thinking, creative direction, problem reframing, "
    "naming, positioning, campaign ideas, business concepts, personal decisions "
    "with creative dimensions, or any challenge that would benefit from multiple "
    "perspectives and structured exploration.\n\n"
    "DIRECT — A straightforward request: a factual question, a simple task, "
    "something that doesn't need the full studio process.\n\n"
    "OUT_OF_SCOPE — Something the studio genuinely cannot help with.\n\n"
    "Reply with ONLY one word: STUDIO, DIRECT, or OUT_OF_SCOPE."
)

routing_response = call_role(
    role="director",
    user_message=routing_task,
    context=initial_prompt,
    blackboard=blackboard,
    model=DIRECTOR_FAST_MODEL,
    max_tokens=20,
    brief_doc=read_brief_doc(session_id),
    provider=PRIMARY_PROVIDER,
)

route = routing_response.strip().upper()
if "STUDIO" in route:
    route = "STUDIO"
elif "DIRECT" in route:
    route = "DIRECT"
else:
    route = "OUT_OF_SCOPE"

session_route = route  # passed to all subsequent cells

scribe_log(
    blackboard,
    "DIRECTOR",
    f"routing_{route.lower()}",
    (
        f"Request routed to {route}. Entering adaptive discovery."
        if route == "STUDIO"
        else f"Request routed to {route}."
    ),
)

print(f"  Route: {route}")
print(f"  Provider: {PRIMARY_PROVIDER} / Critic: {CRITIC_PROVIDER}")

# -- Director opening message to PIL (STUDIO only) ----------------------------
# This is Turn 1 of discovery — the Director's first spoken message.
# Stored as director_message and picked up by Cell 4.
if route == "STUDIO":
    opening_task = (
        f"A person has arrived at The Creative Prism with this:\n\n"
        f'"{initial_prompt}"\n\n'
        "Open a discovery conversation. Write your first message to them.\n\n"
        "Your goal in this first turn is to orient — acknowledge what they "
        "brought, show genuine curiosity, and ask ONE focused question that "
        "surfaces the most important signal you don't yet have.\n\n"
        "Be warm, direct, and conversational. Do not run through a checklist. "
        "Do not explain the process. Just begin.\n\n"
        "One question only. Keep it to 2-4 sentences total."
    )

    director_message = call_role(
        role="director",
        user_message=opening_task,
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=300,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    scribe_log(
        blackboard,
        "DIRECTOR",
        "discovery_opened",
        f"Discovery Turn 1 delivered to PIL.",
    )

    print()
    print("── DIRECTOR ────────────────────────────────────────────────")
    print(director_message)
    print("────────────────────────────────────────────────────────────")

elif route == "DIRECT":
    direct_response = call_role(
        role="director",
        user_message=(
            f'The person asked: "{initial_prompt}"\n\n'
            "This is a direct request — respond helpfully and completely "
            "without engaging the full studio process."
        ),
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=600,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )
    print()
    print(direct_response)
    director_message = direct_response

else:
    out_of_scope_response = (
        "That's outside what The Creative Prism is built to help with. "
        "If you have a creative challenge or strategic problem you'd like "
        "to explore, I'd be glad to help with that."
    )
    print()
    print(out_of_scope_response)
    director_message = out_of_scope_response

print(f"\n✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

if route != "STUDIO":
    print(f"\n⚠ Route is {route} — do not proceed with studio workflow.")

════════════════════════════════════════════════════════════
STAGE 0 — ROUTING
════════════════════════════════════════════════════════════
  Prompt: I have two places I could move to—one feels right but makes no financial sense, ...
  Baseline generated and stored.
  Route: STUDIO
  Provider: openai / Critic: anthropic

── DIRECTOR ────────────────────────────────────────────────
You’re holding a real split here: one option seems to speak to something deeper, and the other makes sense on paper but leaves you cold. Before we sort the places, I want to understand the stakes a little better — if you choose “wrong,” what do you most fear losing: money, momentum, peace, or a part of yourself?
────────────────────────────────────────────────────────────

✓ Reasoning trace: 5 entries


---
## Cell 4 — Stage 1: Adaptive Discovery

Director-controlled discovery loop. One to six turns.
The Director decides when it has enough signal — not the notebook.

Each turn: Director speaks → PIL responds → Director assesses signal.
The Director can deploy extraction techniques (sacrificial concepts,
forced choices, fill-in-the-blank, etc.) when signal is weak.

All PIL interactions are live input.

In [5]:
# ── CELL 4 — STAGE 1: ADAPTIVE DISCOVERY ─────────────────────────────────────

print("=" * 60)
print("STAGE 1 -- ADAPTIVE DISCOVERY")
print("=" * 60)

if session_route != "STUDIO":
    print("Skipped -- session was routed as", session_route)
else:

    MAX_DISCOVERY_TURNS = 6

    # -- Discovery state ---------------------------------------------------
    conversation_history = []
    all_assessments = []
    discovery_status = "CONTINUE"
    current_signals = []
    current_missing = []

    # Turn 1 — Director's opening message was produced in Cell 3
    conversation_history.append({"role": "director", "content": director_message})

    print(f"\n-- Turn 1 -------------------------------------------------")
    print(f"Director: {director_message}\n")

    # -- Collect first PIL response ----------------------------------------
    pil_response = input("> ").strip()
    conversation_history.append({"role": "pil", "content": pil_response})

    # -- Discovery loop ----------------------------------------------------
    turn = 1

    while discovery_status == "CONTINUE" and turn < MAX_DISCOVERY_TURNS:
        turn += 1

        # Format conversation for the Director
        history_formatted = "\n".join(
            [
                ("DIRECTOR: " if e["role"] == "director" else "PERSON: ") + e["content"]
                for e in conversation_history
            ]
        )

        # Ceiling pressure on final turn
        ceiling_note = ""
        if turn == MAX_DISCOVERY_TURNS:
            ceiling_note = (
                f"\n\nThis is turn {turn} of {MAX_DISCOVERY_TURNS} -- your last turn. "
                "Set status to READY and work with the signal you have."
            )

        discovery_task = (
            "You are the Director of The Creative Prism, in a discovery "
            "conversation with a person who has brought a creative challenge.\n\n"
            f"CONVERSATION SO FAR:\n{history_formatted}\n\n"
            f"TURN: {turn} of {MAX_DISCOVERY_TURNS}{ceiling_note}\n\n"
            "YOUR EXTRACTION TOOLKIT (use when signal is weak or vague):\n"
            "- Forced Choice: Does this feel more like A or B?\n"
            "- Elimination: Which of these feels least right?\n"
            "- Extremes Test: Push concept to an extreme, observe comfort\n"
            "- Reaction Snapshot: Quick gut reaction, before you think about it?\n"
            "- Fill in the Blank: This should feel more like ___ than ___\n"
            "- Analogy Probe: If this were a place/object/experience, what would it be?\n"
            "- Constraint Introduction: If you had to explain in one sentence, what matters most?\n"
            "- Signal Amplification: Reflect what you heard slightly amplified, invite correction\n"
            "- Micro-Ranking: Rank these 1-3 on instinct: X / Y / Z\n"
            "- Sacrificial Concept: Present a deliberately wrong/strange idea -- "
            "the value is in what the person pushes back against and why\n"
            "- Category Transplant: Propose the solution as if it belonged to "
            "a different industry entirely\n\n"
            "GUIDANCE:\n"
            "- After turn 2, if the person gives broad or vague answers, DEPLOY a specific\n"
            "  extraction technique. Do not ask another open-ended question. Use a forced\n"
            "  choice, fill-in-the-blank, sacrificial concept, or analogy probe instead.\n"
            "- Offer specific alternatives rather than asking how they see things.\n"
            "- The person may not be experienced with AI. Meet them where they are.\n"
            "- Vague or incomplete answers are normal -- your toolkit handles this.\n"
            "- Acknowledge what they said before your next question. Brief, genuine.\n"
            "- ONE question or probe per turn. Never stack multiple questions.\n"
            "- Use extraction techniques when standard questions produce vague signal.\n"
            "- You are trying to understand the PERSON, not just the problem.\n"
            "- Keep it conversational -- a smart, engaged collaborator.\n\n"
            "WHAT YOU NEED FOR A CREATIVE BRIEF:\n"
            "- The challenge: what they are actually trying to solve\n"
            "- The context: who it is for, where it lives, what the situation is\n"
            "- What success looks like: desired outcome\n"
            "- What to avoid: boundaries, anti-patterns, negative space\n"
            "- The person: what they value beneath what they have said\n\n"
            "CRITICAL RULE FOR STATUS:\n"
            "- If you set status to READY, your message must be a CLOSING STATEMENT,\n"
            "  not a question. Summarize what you have learned and signal that you are\n"
            "  about to move to the next phase. Do NOT ask a question and set READY.\n"
            "- If you still have a question to ask, set status to CONTINUE.\n\n"
            "RESPOND IN TWO PARTS:\n\n"
            "PART 1: Your message to the person. Conversational, warm, one "
            "question or probe. Acknowledge what they said first.\n\n"
            "---SIGNAL_ASSESSMENT---\n\n"
            "PART 2: A JSON signal assessment (the person will NOT see this):\n"
            "{\n"
            '  "signals_gathered": ["list each distinct signal established so far"],\n'
            '  "signals_missing": ["what you still need to understand"],\n'
            '  "technique_used": "none | forced_choice | elimination | extremes_test | '
            "reaction_snapshot | fill_blank | analogy_probe | constraint_intro | "
            'signal_amplification | micro_ranking | sacrificial_concept | category_transplant",\n'
            '  "status": "CONTINUE | READY",\n'
            '  "reason": "one sentence -- why you need more or why you have enough"\n'
            "}"
        )

        director_response = call_role(
            role="director",
            user_message=discovery_task,
            context=initial_prompt,
            blackboard=blackboard,
            model=DIRECTOR_MODEL,
            max_tokens=600,
            brief_doc=read_brief_doc(session_id),
            provider=PRIMARY_PROVIDER,  # ← hybrid: use primary provider
        )

        # -- Parse the two-part response -----------------------------------
        if "---SIGNAL_ASSESSMENT---" in director_response:
            parts = director_response.split("---SIGNAL_ASSESSMENT---", 1)
            director_message = parts[0].strip()
            assessment_raw = parts[1].strip()
        else:
            director_message = director_response.strip()
            assessment_raw = json.dumps(
                {
                    "signals_gathered": current_signals,
                    "signals_missing": ["assessment delimiter not found"],
                    "technique_used": "none",
                    "status": "CONTINUE" if turn < MAX_DISCOVERY_TURNS else "READY",
                    "reason": "Signal assessment delimiter not found in response",
                }
            )

        # Parse signal assessment JSON
        try:
            clean_a = assessment_raw.strip()
            if clean_a.startswith("```"):
                clean_a = clean_a.split("```")[1]
                if clean_a.startswith("json"):
                    clean_a = clean_a[4:]
            brace_depth = 0
            json_end = 0
            for ci, ch in enumerate(clean_a):
                if ch == "{":
                    brace_depth += 1
                elif ch == "}":
                    brace_depth -= 1
                    if brace_depth == 0:
                        json_end = ci + 1
                        break
            if json_end > 0:
                clean_a = clean_a[:json_end]
            assessment = json.loads(clean_a.strip())
        except (json.JSONDecodeError, ValueError, IndexError):
            assessment = {
                "signals_gathered": current_signals,
                "signals_missing": ["assessment parsing failed"],
                "technique_used": "unknown",
                "status": "CONTINUE" if turn < MAX_DISCOVERY_TURNS else "READY",
                "reason": "JSON parsing failed",
            }

        discovery_status = assessment.get("status", "CONTINUE").upper()
        current_signals = assessment.get("signals_gathered", current_signals)
        current_missing = assessment.get("signals_missing", [])
        technique = assessment.get("technique_used", "none")
        reason = assessment.get("reason", "")
        all_assessments.append(assessment)

        conversation_history.append({"role": "director", "content": director_message})

        # -- Display -------------------------------------------------------
        print(f"-- Turn {turn} -------------------------------------------------")
        print(f"Director: {director_message}\n")
        if technique and technique != "none":
            print(f"  [technique: {technique}]")
        print(f"  [status: {discovery_status} | {reason}]")
        print()

        if discovery_status == "READY":
            # Safety net: if Director asked a question while setting READY
            if director_message.rstrip().endswith("?"):
                print("  [Director asked a question with READY — collecting response]")
                pil_response = input("> ").strip()
                conversation_history.append({"role": "pil", "content": pil_response})
            break

        # -- Collect PIL response ------------------------------------------
        pil_response = input("> ").strip()
        conversation_history.append({"role": "pil", "content": pil_response})

    # -- Store discovery in Blackboard and Brief Document ------------------
    blackboard["discovery"]["orientation_summary"] = initial_prompt
    blackboard["discovery"]["context_insights"] = current_signals
    blackboard["discovery"]["preferences"] = [
        s
        for s in current_signals
        if any(
            kw in s.lower()
            for kw in ["avoid", "prefer", "want", "don't", "hate", "love", "feel"]
        )
    ]

    # Check if sacrificial concept was used
    for idx, a in enumerate(all_assessments):
        if a.get("technique_used") == "sacrificial_concept":
            dir_idx = (idx + 1) * 2
            if dir_idx < len(conversation_history):
                probe = conversation_history[dir_idx].get("content", "")
                resp_idx = dir_idx + 1
                resp = (
                    conversation_history[resp_idx].get("content", "")
                    if resp_idx < len(conversation_history)
                    else ""
                )
                blackboard["discovery"]["sacrificial_exploration"][
                    "probe_prompt"
                ] = probe
                blackboard["discovery"]["sacrificial_exploration"][
                    "user_response"
                ] = resp
                blackboard["discovery"]["sacrificial_exploration"][
                    "insight_extracted"
                ] = f"Sacrificial concept deployed at turn {idx+2}. PIL reaction recorded."
            break

    # Record techniques used
    techniques_used = [
        a.get("technique_used", "none")
        for a in all_assessments
        if a.get("technique_used", "none") != "none"
    ]
    blackboard["discovery"]["exploratory_prompts"] = techniques_used

    # Store full conversation in Studio Brief Document
    discovery_log = "\n\n".join(
        [
            ("**Director:** " if e["role"] == "director" else "**Person:** ")
            + e["content"]
            for e in conversation_history
        ]
    )
    update_brief_doc(session_id, "DIRECTOR", "DISCOVERY CONVERSATION", discovery_log)

    if current_signals:
        signals_section = "\n".join(f"- {s}" for s in current_signals)
        update_brief_doc(session_id, "DIRECTOR", "SIGNALS GATHERED", signals_section)

    if current_missing:
        missing_section = "\n".join(f"- {m}" for m in current_missing)
        update_brief_doc(session_id, "DIRECTOR", "OPEN SIGNAL GAPS", missing_section)

    scribe_log(
        blackboard,
        "DIRECTOR",
        "discovery_complete",
        f"Adaptive discovery complete. {turn} turns. "
        f"{len(current_signals)} signals gathered. "
        f"Techniques: {techniques_used if techniques_used else 'none'}.",
    )

    print(f"\nDiscovery complete -- {turn} turns")
    print(f"Signals gathered: {len(current_signals)}")
    if techniques_used:
        print(f"Techniques used: {', '.join(techniques_used)}")
    if current_missing:
        print(f"Still missing: {', '.join(current_missing[:3])}")
    print(f"Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

STAGE 1 -- ADAPTIVE DISCOVERY

-- Turn 1 -------------------------------------------------
Director: You’re holding a real split here: one option seems to speak to something deeper, and the other makes sense on paper but leaves you cold. Before we sort the places, I want to understand the stakes a little better — if you choose “wrong,” what do you most fear losing: money, momentum, peace, or a part of yourself?

-- Turn 2 -------------------------------------------------
Director: That helps. Quick gut check: is the bigger danger here **becoming financially strained**, or **ending up somewhere that slowly drains you**?

  [technique: forced_choice]
  [status: CONTINUE | Their last answer confirmed the dilemma but added no new signal, so a forced choice is needed to surface the dominant underlying fear.]

-- Turn 3 -------------------------------------------------
Director: Got it — so the problem isn’t that one is obviously “the risky one” and the other is obviously “the safe one.” It’

---
## Cell 5 — Stage 2 & 3: Reframing + Creative Brief

The Director proposes a reframing of the challenge to the PIL for confirmation.
This is the moment where the PIL sees whether the Director understood them.
After confirmation, the Director produces the structured creative brief.

The reframing step closes the discovery loop by
demonstrating understanding before the Creative Team is engaged.

In [6]:
# -- STAGE 2: REFRAMING ----------------------------------------------------
# -- STAGE 3: CREATIVE BRIEF -----------------------------------------------

print("=" * 60)
print("STAGE 2 -- REFRAMING")
print("STAGE 3 -- CREATIVE BRIEF")
print("=" * 60)

if session_route != "STUDIO":
    print("Skipped -- session was routed as", session_route)
else:

    # -- Build discovery summary from conversation -------------------------
    discovery_log_plain = "\n".join(
        [
            ("Director: " if e["role"] == "director" else "Person: ") + e["content"]
            for e in conversation_history
        ]
    )

    signals_summary = (
        "\n".join(f"- {s}" for s in current_signals)
        if current_signals
        else "None captured"
    )
    missing_summary = (
        "\n".join(f"- {m}" for m in current_missing) if current_missing else "None"
    )

    # -- Director proposes reframing ---------------------------------------
    reframe_task = (
        "You have just completed a discovery conversation. Now propose a "
        "reframing of the person's challenge.\n\n"
        f"DISCOVERY CONVERSATION:\n{discovery_log_plain}\n\n"
        f"SIGNALS GATHERED:\n{signals_summary}\n\n"
        f"GAPS REMAINING:\n{missing_summary}\n\n"
        "Your reframing should:\n"
        "- Show the person you understood not just what they said, but what they meant\n"
        "- Restate the challenge in a way that is sharper than how they put it\n"
        "- Surface any insight from the conversation they may not have seen themselves\n"
        "- Be 3-5 sentences -- clear, confident, specific to this person\n\n"
        "End by asking the person to confirm this is right, or to adjust anything "
        "that is off. This is the moment where they see whether you understood them."
    )

    reframe_response = call_role(
        role="director",
        user_message=reframe_task,
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=400,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    print("-- DIRECTOR REFRAMING ------------------------------------------")
    print(reframe_response)
    print()

    # -- PIL confirms or adjusts -------------------------------------------
    pil_confirmation = input("> ").strip()

    # -- Director reads PIL response -- branch on content ------------------
    signal_read_task = (
        f"The PIL has responded to your reframing:\n\n"
        f"YOUR REFRAMING:\n{reframe_response}\n\n"
        f"PIL RESPONSE:\n{pil_confirmation}\n\n"
        "Read this carefully. Determine which of these applies:\n\n"
        "A) CONFIRMED — the PIL agreed with the reframing, possibly with minor affirmation. "
        "Respond with one brief, natural acknowledgment (one sentence) and nothing else.\n\n"
        "B) NEW SIGNAL — the PIL added new information, corrected something, or shifted "
        "the framing. Respond by: (1) acknowledging what they added in one sentence, "
        "(2) stating how it changes your understanding of the challenge in one sentence. "
        "Do not re-present the full reframing. Just absorb and confirm the addition.\n\n"
        "C) SIGNIFICANT CORRECTION — the PIL said the reframing was substantially off. "
        "Respond by acknowledging the correction and asking ONE focused question to "
        "clarify what you missed. One sentence acknowledgment, one question.\n\n"
        "Keep your response to 2-3 sentences maximum. "
        "Do not repeat content from the reframing. Do not ask multiple questions."
    )

    signal_read_response = call_role(
        role="director",
        user_message=signal_read_task,
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_FAST_MODEL,
        max_tokens=150,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    print()
    print(signal_read_response)
    print()

    # -- Store reframing exchange including Director acknowledgment --------
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "REFRAMING",
        f"**Director:**\n{reframe_response}\n\n"
        f"**Person:**\n{pil_confirmation}\n\n"
        f"**Director acknowledged:**\n{signal_read_response}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "reframing_confirmed",
        f"Reframing confirmed. Director acknowledged: {signal_read_response[:80]}",
    )

    blackboard["discovery"]["desired_outcome"] = reframe_response

    # -- Director synthesizes creative brief -------------------------------
    brief_task = (
        "Based on this confirmed understanding, produce a structured creative brief.\n\n"
        f"DISCOVERY CONVERSATION:\n{discovery_log_plain}\n\n"
        f"REFRAMING (confirmed by person):\n{reframe_response}\n\n"
        f"PERSON'S RESPONSE TO REFRAMING:\n{pil_confirmation}\n\n"
        f"DIRECTOR'S READING OF THAT RESPONSE:\n{signal_read_response}\n\n"
        f"SIGNALS GATHERED:\n{signals_summary}\n\n"
        "Return ONLY a JSON object -- no preamble, no markdown fences:\n\n"
        "{\n"
        '  "challenge": "one clear sentence -- the actual challenge, not the stated one",\n'
        '  "context": "2-3 sentences about situation, audience, and what matters",\n'
        '  "desired_result": "what success looks like -- specific to what was revealed",\n'
        '  "constraints": ["constraint 1", "constraint 2"],\n'
        '  "pressure_point": "one sentence naming the productive tension or paradox the Creative Team should press against -- not resolve, press against",\n'
        '  "assumptions": ["assumption the Director is making that the PIL should validate or correct", "assumption 2"]\n'
        "}\n\n"
        "Be specific. This brief becomes the Creative Team's working document. "
        "Include what the person explicitly said AND what their reactions revealed. "
        "The pressure_point is the creative instrument -- it names what makes this "
        "problem genuinely open rather than already solved. "
        "The assumptions are things the Director inferred that the PIL should "
        "confirm or correct -- not questions for the PIL to answer, but premises "
        "the Director is building on."
    )

    brief_response = call_role(
        role="director",
        user_message=brief_task,
        context=initial_prompt,
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=800,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    # -- Parse brief -- JSON with validation gate --------------------------
    brief = blackboard["creative_brief"]

    try:
        clean = brief_response.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        parsed = json.loads(clean.strip())

        brief["challenge"] = parsed.get("challenge", "")
        brief["context"] = parsed.get("context", "")
        brief["desired_result"] = parsed.get("desired_result", "")
        brief["constraints"] = parsed.get("constraints", [])
        brief["pressure_point"] = parsed.get("pressure_point", "")
        brief["open_questions"] = parsed.get(
            "assumptions", []
        )  # stored as open_questions for compatibility

        missing = [
            f
            for f in ["challenge", "context", "desired_result"]
            if not brief.get(f, "").strip()
        ]
        if missing:
            raise ValueError(f"Brief missing required fields: {missing}")

        print("Brief parsed successfully")

    except (json.JSONDecodeError, ValueError) as e:
        print(f"Brief parsing failed: {e}")
        print("Raw response:")
        print(brief_response)
        raise RuntimeError(
            "Brief parsing failed -- session cannot continue with an incomplete brief.\n"
            "Review the raw response above and re-run this cell."
        )

    # -- Store and display -------------------------------------------------
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "CREATIVE BRIEF",
        f"**Challenge:** {brief['challenge']}\n\n"
        f"**Context:** {brief['context']}\n\n"
        f"**Desired result:** {brief['desired_result']}\n\n"
        f"**Constraints:** {', '.join(brief['constraints'])}\n\n"
        f"**Pressure point:** {brief.get('pressure_point', '')}\n\n"
        f"**Assumptions to validate:** {', '.join(brief['open_questions'])}",
    )

    scribe_log(
        blackboard,
        "DIRECTOR",
        "brief_synthesized",
        f"Creative brief produced. Challenge: '{brief['challenge'][:60]}...'",
    )
    validate_stage(blackboard, "brief")

    print()
    print(f"Challenge      : {brief['challenge']}")
    print(f"Context        : {brief['context'][:80]}...")
    print(f"Desired result : {brief['desired_result'][:80]}...")
    print(f"Constraints    : {brief['constraints']}")
    print(f"Pressure point : {brief.get('pressure_point', '')[:80]}")
    print(f"Assumptions    : {len(brief['open_questions'])}")
    print()
    print(f"Creative brief complete")
    print(f"Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

STAGE 2 -- REFRAMING
STAGE 3 -- CREATIVE BRIEF
-- DIRECTOR REFRAMING ------------------------------------------
This doesn’t sound like a choice between a “good” option and a “bad” one — it sounds like a choice between two different kinds of regret. One place risks putting pressure on your finances in exchange for feeling more like you, while the other protects practicality but may leave you feeling like you settled into a life that doesn’t really fit. The hard part is that you’re not just choosing where to live; you’re choosing which kind of cost is more tolerable: external strain or internal drain. So the real decision is which option gives you the stronger chance of building a life you can sustain without quietly resenting it later.

Does that feel right, or is there something important I’m missing or misreading?


That’s a sharp correction.  
You’re not choosing between two regrettable tradeoffs so much as noticing that both options feel like a trap, which means we need to look for

---
## Cell 5a — Stage 3a: Team Configuration

The Director configures the Creative Team's personality traits
for this specific problem. Adjusts 9 core trait weights and writes
a short team brief for each role (Creator, Critic, Researcher).

This is invisible to the PIL — it is the Director selecting the
right team for the job, exactly as a Creative Director would in
a real agency. The team brief is written to the Studio Brief Document
so every role reads it before acting.

In [7]:
# -- STAGE 3a: TEAM CONFIGURATION ------------------------------------------

print("=" * 60)
print("STAGE 3a -- TEAM CONFIGURATION")
print("=" * 60)

if session_route != "STUDIO":
    print("Skipped")
else:
    tunable_summary = ""
    for role_name in ["creator", "critic", "researcher"]:
        tunable = get_tunable_traits(role_name, traits_matrix)
        tunable_summary += "\n" + role_name.upper() + " tunable traits:\n"
        for t in tunable:
            tunable_summary += (
                "  "
                + t["trait"]
                + " (base: "
                + f"{t['base']:.2f}, range: {t['min']:.2f}-{t['max']:.2f})"
                f" -- {t['description']}\n"
            )

    config_task = (
        "You are the Director. The brief is complete. Configure the Creative "
        "Team for this specific problem.\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        "Your job is not to select traits that will help the team handle this "
        "problem competently. Your job is to configure a team that will push "
        "past the expected answer — toward directions the PIL would not have "
        "predicted.\n\n"
        "Think in combinations and tensions, not individual dials. A Creator "
        "with maximum imagination paired with maximum iconoclasm will challenge "
        "the premise of the brief. A Critic with maximum skepticism alongside "
        "maximum intellectual_generosity will destroy weak ideas while building "
        "on strong ones. Ask not what traits fit this domain — ask what "
        "perspective would genuinely surprise this problem.\n\n"
        "REQUIREMENTS:\n"
        "- Minimum 8 traits adjusted per role\n"
        "- At least 3 traits per role pushed to or near their maximum allowed value\n"
        "- Include traits from at least 3 different categories per role\n"
        "- Avoid defaulting to the obvious traits for the obvious domain\n\n"
        f"TUNABLE TRAITS (use verbatim names only):\n{tunable_summary}\n\n"
        "Return ONLY JSON -- no preamble:\n"
        "{\n"
        '  "creator_adjustments": {"trait_name": value, ...},\n'
        '  "critic_adjustments": {"trait_name": value, ...},\n'
        '  "researcher_adjustments": {"trait_name": value, ...},\n'
        '  "rationale": "2-3 sentences naming the creative intention, not just the problem type"\n'
        "}\n\n"
        "Only list traits you want to CHANGE from base. Unmentioned traits stay "
        "at their base values. Values must be within the allowed range. "
        "CRITICAL: trait names must match the list above exactly, character for character. "
        "Do not invent, abbreviate, or paraphrase trait names. "
        "Any name not found verbatim in the list will be discarded."
    )

    config_response = call_role(
        role="director",
        user_message=config_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=900,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    try:
        clean = config_response.strip()
        if clean.startswith("```"):
            clean = clean.split("```")[1]
            if clean.startswith("json"):
                clean = clean[4:]
        brace_depth = 0
        json_end = 0
        for ci, ch in enumerate(clean):
            if ch == "{":
                brace_depth += 1
            elif ch == "}":
                brace_depth -= 1
                if brace_depth == 0:
                    json_end = ci + 1
                    break
        if json_end > 0:
            clean = clean[:json_end]
        config = json.loads(clean.strip())

        for role_name in ["creator", "critic", "researcher"]:
            raw = config.get(role_name + "_adjustments", {})
            validated, warnings = validate_adjustments(role_name, raw, traits_matrix)
            session_adjustments[role_name] = validated
            for w in warnings:
                print(f"  Warning: {w}")

        for role_name in ["creator", "critic", "researcher"]:
            trait_profiles[role_name] = build_trait_profile(
                role_name, session_adjustments[role_name], traits_matrix
            )

        rationale = config.get("rationale", "")
        team_brief = "**Director reasoning:** " + rationale + "\n"
        for role_name in ["creator", "critic", "researcher"]:
            adj = session_adjustments[role_name]
            if adj:
                adj_str = ", ".join(f"{k}: {v:.2f}" for k, v in adj.items())
                team_brief += "\n**" + role_name.upper() + " adjustments:** " + adj_str

        update_brief_doc(session_id, "DIRECTOR", "TEAM CONFIGURATION", team_brief)
        scribe_log(
            blackboard,
            "DIRECTOR",
            "team_configured",
            f"Team configured. {rationale[:80]}...",
        )

        print(f"Team configured.\n\nRationale: {rationale}\n")
        for role_name in ["creator", "critic", "researcher"]:
            adj = session_adjustments[role_name]
            n = len(adj)
            print(
                f"{role_name.upper()}: {n} traits adjusted"
                + (f" -- {adj}" if adj else "")
            )

    except (json.JSONDecodeError, ValueError) as e:
        print(f"Team config parsing failed: {e}. Using default traits.")
        for role_name in ["creator", "critic", "researcher"]:
            trait_profiles[role_name] = build_trait_profile(
                role_name, {}, traits_matrix
            )

STAGE 3a -- TEAM CONFIGURATION
Team configured.

Rationale: I’m configuring a team to refuse the false dignity of a tragic tradeoff and instead hunt for the hidden variable that changes the geometry of the decision. The Creator is pushed toward premise-breaking reframes, the Critic is tuned to aggressively test whether those reframes are real rather than consoling, and the Researcher is aimed at finding patterns from adjacent domains where seemingly binary life choices became workable through redesign rather than sacrifice.

CREATOR: 13 traits adjusted -- {'imagination': 0.95, 'iconoclasm': 0.85, 'reframing': 0.88, 'beginner_mind': 0.88, 'perspective_shifting': 0.85, 'experimentalism': 0.88, 'risk_taking': 0.85, 'playfulness': 0.82, 'inventiveness': 0.95, 'systems_thinking': 0.7, 'synthesis': 0.75, 'pragmatism': 0.58, 'respectful_challenge': 0.74}
CRITIC: 13 traits adjusted -- {'skepticism': 0.95, 'rigor': 0.95, 'intellectual_generosity': 0.72, 'constructive_feedback': 0.88, 'respectfu

---
## Cell 5b — Stage 3b: Assumption Validation 

Director validates assumptions from the brief to the PIL before ideation begins.
Answers shape the brief and the Studio Brief Document.
PIL responds to open questions via live input.

In [8]:
# ── STAGE 3b: ASSUMPTION VALIDATION ──────────────────────────
# v3.1 patch: assumptions now numbered (1. 2. 3.) so PIL can respond by number

print("═" * 60)
print("STAGE 3b — ASSUMPTION CHECK")
print("═" * 60)

assumptions = brief.get(
    "open_questions", []
)  # stored as open_questions for schema compatibility

if not assumptions:
    print("✓ No assumptions to validate — proceeding to ideation")
else:
    # -- Director presents assumptions for validation
    assumption_task = (
        f"The creative brief is complete. Before the Creative Team begins work, "
        f"surface the assumptions you made during briefing so the PIL can correct "
        f"any that are wrong.\n\n"
        f"ASSUMPTIONS TO VALIDATE:\n"
        + "\n".join(f"{i+1}. {a}" for i, a in enumerate(assumptions))
        + "\n\n"
        f"BRIEF CHALLENGE: {brief['challenge']}\n\n"
        f"Present these as a numbered list (1. 2. 3.) — keep the numbers from above. "
        f"These are premises you built on, things you inferred and want to confirm. "
        f"Add one final item to the numbered list — number {len(assumptions) + 1} — "
        f"asking what would be completely off the table for this work. "
        f"Do NOT ask the PIL to answer strategic questions or do creative work. "
        f"You are asking them to correct wrong assumptions and name hard limits. "
        f"Keep it conversational and specific. One to three sentences per item."
    )

    assumption_response = call_role(
        role="director",
        user_message=assumption_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_FAST_MODEL,
        max_tokens=350,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    print("── DIRECTOR ────────────────────────────────────────────────")
    print(assumption_response)
    print()

    pil_answers = input("> ").strip()

    # -- Director synthesizes PIL response into brief additions
    synthesis_task = (
        f"The PIL has responded to your assumption check:\n\n"
        f"YOUR ASSUMPTIONS:\n{assumption_response}\n\n"
        f"PIL RESPONSE:\n{pil_answers}\n\n"
        f"CURRENT BRIEF CHALLENGE: {brief['challenge']}\n\n"
        f"Do three things:\n"
        f"1. Acknowledge what the PIL said in one warm, specific sentence — "
        f"name what they corrected or confirmed, so they know they were heard.\n"
        f"2. State in plain language: what 1-3 things does this add or change in "
        f"the brief? Frame these as additions the Creative Team will now work with.\n"
        f"3. Confirm you are ready to begin.\n\n"
        f"Keep it to 3-4 sentences total. Do not ask follow-up questions. "
        f"The PIL should feel heard and should understand exactly what changed."
    )

    synthesis_response = call_role(
        role="director",
        user_message=synthesis_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_FAST_MODEL,
        max_tokens=200,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    print(synthesis_response)
    print()

    # -- Store everything and enrich brief
    brief["open_questions"].append(f"PIL answered: {pil_answers}")
    brief["open_questions"].append(f"Director synthesis: {synthesis_response}")

    update_brief_doc(
        session_id,
        "DIRECTOR",
        "ASSUMPTION VALIDATION",
        f"**Director presented assumptions:**\n{assumption_response}\n\n"
        f"**PIL responded:**\n{pil_answers}\n\n"
        f"**Director synthesized and acknowledged:**\n{synthesis_response}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "open_questions_answered",
        f"Assumption validation complete. Brief enriched. PIL acknowledged: {synthesis_response[:80]}",
    )

    print("✓ Assumptions validated — brief enriched")
    print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 3b — ASSUMPTION CHECK
════════════════════════════════════════════════════════════
── DIRECTOR ────────────────────────────────────────────────
Absolutely — before I hand this to the Creative Team, I want to make sure I’m not building on the wrong premises.

1. **The financially impractical option feels “right” for reasons substantial enough to affect your sense of identity, belonging, or aliveness — not just because it’s a passing preference.** If that’s not true, the direction of the brief changes a lot.

2. **The practical option feels empty in a way that could become psychologically costly over time, not just disappointing in the short term.** I’m assuming this is about more than aesthetics.

3. **When you say you’d regret either choice, I’m taking that as a real signal that both options feel structurally incomplete — not just that you’re indecisive or stuck.** If that’s off, I should adjust the framing.

4. **I’m a

---
## Cell 6a — Stage 4a: Creator

The creative brief goes directly to the Creator first — not the Researcher.
The Creator generates candidate directions informed by the brief.
Verbalized Sampling applied: probability weights push toward lateral thinking.
The Researcher has not yet acted — the Creative Team works first.

In [9]:
# ── STAGE 4a: CREATOR — IDEATION PASS 1 ──────────────────────────────────────

print("═" * 60)
print("STAGE 4a — CREATOR: IDEATION (PASS 1)")
print("═" * 60)

creator_task = f"""You are working on this creative brief:

CHALLENGE: {brief['challenge']}
CONTEXT: {brief['context']}
DESIRED RESULT: {brief['desired_result']}
CONSTRAINTS: {", ".join(brief['constraints'])}

Generate 3 distinct conceptual directions using Verbalized Sampling.

━━━ VERBALIZED SAMPLING — ASSIGN THE BAND FIRST, THEN WRITE THE DIRECTION ━━━

One direction per band. Assign the band before writing. No two directions share a band.
The band determines the kind of thinking required — not a label applied afterward.

HIGH BAND (0.55–0.80)
The intelligent expected answer. A skilled practitioner working this brief carefully
would likely arrive here. Credible, grounded, implementable. Not obvious — but not
surprising. This is the direction that earns trust.

MID BAND (0.25–0.50)
A genuine reframe. The same problem understood differently. Requires stepping outside
the category's conventional assumptions. Not a variation on the high band direction —
a different way of understanding what the brief is actually asking for.

LOW BAND (0.08–0.22)
The direction that arrives from outside the expected territory entirely. Precise and
specific — lateral thinking is not loose thinking. This direction should make the room
go quiet. Uncomfortable and exact. A conventional approach would rarely reach here.

Assign your score within the band honestly. If it feels like 0.14, say 0.14.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

For each direction:
IDEA_ID: [A1, A2, A3]
BAND: [HIGH / MID / LOW]
PROBABILITY: [score within band]
TITLE: [short evocative name]
CONCEPT: [2-3 sentences — what is this direction?]
RATIONALE: [why this is worth exploring]
EMOTIONAL TERRITORY: [what feeling does this occupy?]
RESEARCH_REQUEST: [optional — one specific research question if external knowledge
would materially strengthen or ground this direction. Omit entirely if not needed.]

Do not write any preamble or analysis before your directions. Start directly with IDEA_ID: A1.

Be genuinely distinct — not three variations of the same idea."""

creator_response = call_role(
    role="creator",
    user_message=creator_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=1800,
    trait_profile=trait_profiles["creator"],
    brief_doc=read_brief_doc(session_id),
    provider=PRIMARY_PROVIDER,
)

# Store as cycle 1
cycle_1 = {
    "cycle_number": 1,
    "creator_proposals": [{"raw_response": creator_response}],
    "critic_feedback": [],
    "synthesized_directions": [],
}
blackboard["ideation_cycles"].append(cycle_1)
update_brief_doc(session_id, "CREATOR", "DIRECTIONS — PASS 1", creator_response)
scribe_log(
    blackboard,
    "CREATOR",
    "ideation_complete",
    "Three directions generated using banded Verbalized Sampling. Cycle 1 logged.",
)
validate_stage(blackboard, "ideation")

print("── CREATOR DIRECTIONS ──────────────────────────────────────")
print(creator_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Creator pass 1 complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4a — CREATOR: IDEATION (PASS 1)
════════════════════════════════════════════════════════════
── CREATOR DIRECTIONS ──────────────────────────────────────
IDEA_ID: A1  
BAND: HIGH  
PROBABILITY: 0.68  
TITLE: Make the “right” place affordable by changing the move, not the desire  
CONCEPT: Treat the emotionally right place as the preferred anchor, but test whether its financial unsoundness is actually a property of the place or of the current version of the move. This direction looks for redesign levers: smaller unit, different neighborhood, roommate, temporary sublet, later timing, income adjustment, or a shorter commitment that preserves the core feeling without forcing a permanent financial sacrifice.  
RATIONALE: This is the most grounded path because it respects the emotional truth of the right place while refusing to accept the budget as fixed until the move has been stress-tested. If a workable version exists, it 

---
## Cell 6b — Stage 4b: Critic

The Critic evaluates Creator directions and proposes a synthesis.
Reads the full Studio Brief Document including Creator output.
Constructive pressure — not rejection.

In [10]:
# ── CELL 6b — STAGE 4b: CRITIC — EVALUATION PASS 1 (Hybrid) ──────────────────
# Change from v3_1: call_role() now passes provider=CRITIC_PROVIDER
# and model=CRITIC_MODEL so the Critic runs on the opposite provider.
# engine_hybrid.build_prompt() automatically loads critic_gpt.md
# when provider="openai", critic.md when provider="anthropic".
# Everything else is identical to the v3_1 cell.

print("═" * 60)
print("STAGE 4b — CRITIC: EVALUATION (PASS 1)")
print(f"  Critic provider: {CRITIC_PROVIDER} / {CRITIC_MODEL}")
print("═" * 60)

creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][0][
    "raw_response"
]

critic_task = f"""The Creator has proposed three directions for the brief:

BRIEF CHALLENGE: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

CREATOR'S DIRECTIONS:
{creator_output}

Evaluate each direction using this tight format:

IDEA_ID: [matching the Creator's ID]
WHAT HOLDS: [1-2 sentences — what genuinely works and why]
WHAT DOESN'T: [1-2 sentences — the specific weakness or risk]
REFINEMENT PATH: [one concrete action that would strengthen this direction]
RESEARCH_REQUEST: [optional — one specific research question if external knowledge
would materially help evaluate or strengthen this direction. Omit if not needed.]

After all three evaluations, note any cross-direction patterns if present:
PATTERN: [optional — if multiple directions share a common strength or weakness worth naming]
Present all directions [A1, A2, A3] with their trade-offs stated factually -- do not rank or recommend.

Be rigorous and efficient. Identify the sharpest refinement path for each direction."""

critic_response = call_role(
    role="critic",
    user_message=critic_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=CRITIC_MODEL,
    max_tokens=2400,
    trait_profile=trait_profiles["critic"],
    brief_doc=read_brief_doc(session_id),
    provider=CRITIC_PROVIDER,  # ← hybrid change
)

blackboard["ideation_cycles"][-1]["critic_feedback"].append(
    {"raw_response": critic_response}
)
update_brief_doc(session_id, "CRITIC", "EVALUATION — PASS 1", critic_response)
scribe_log(
    blackboard,
    "CRITIC",
    "evaluation_complete",
    f"Pass 1 evaluation complete [{CRITIC_PROVIDER}]. Refinement paths identified.",
)
validate_stage(blackboard, "critic")

# Parse RESEARCH_REQUESTs from Creator and Critic for Researcher routing
creator_requests = re.findall(r"RESEARCH_REQUEST:\s*(.+?)(?=\n|$)", creator_output)
critic_requests = re.findall(r"RESEARCH_REQUEST:\s*(.+?)(?=\n|$)", critic_response)
research_requests = [
    f"[Creator]: {r.strip()}" for r in creator_requests if r.strip()
] + [f"[Critic]:  {r.strip()}" for r in critic_requests if r.strip()]

print("── CRITIC EVALUATION ───────────────────────────────────────")
print(critic_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Critic pass 1 complete [{CRITIC_PROVIDER}]")
if research_requests:
    print(f"  RESEARCH_REQUESTs found: {len(research_requests)}")
    for r in research_requests:
        print(f"    {r}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4b — CRITIC: EVALUATION (PASS 1)
  Critic provider: anthropic / claude-haiku-4-5-20251001
════════════════════════════════════════════════════════════
── CRITIC EVALUATION ───────────────────────────────────────
# CRITIC EVALUATION — PASS 1
*CRITIC — 17:42*

---

## IDEA_ID: A1

**WHAT HOLDS:**
This direction respects the emotional truth of the right place while refusing to treat the budget as a fixed constraint — it assumes the financial barrier is a property of the *current move design* rather than the place itself. The redesign levers (smaller unit, roommate, timing, income adjustment) are concrete and testable, which means this path creates actionable next steps rather than abstract deliberation.

**WHAT DOESN'T:**
The direction assumes that redesigning the move around the preferred place is possible without eliminating what makes it feel "right." But the PIL's feeling of emptiness about the practical option suggest

In [11]:
print("ideation_cycles count:", len(blackboard["ideation_cycles"]))
last = blackboard["ideation_cycles"][-1]
print("Keys in last cycle:", list(last.keys()))
print("creator_proposals count:", len(last.get("creator_proposals", [])))
print("critic_feedback count:", len(last.get("critic_feedback", [])))

ideation_cycles count: 1
Keys in last cycle: ['cycle_number', 'creator_proposals', 'critic_feedback', 'synthesized_directions']
creator_proposals count: 1
critic_feedback count: 1


---
## Cell 7 — Stage 4c: Researcher

The Researcher acts AFTER the Creative Team — not before.
It reads the full Studio Brief Document including the ideation cycle
and provides contextual enrichment informed by what the team actually produced.
This is also when the bounded autonomous contribution trigger is evaluated:
the Researcher now has sufficient thinking on the Blackboard to identify genuine gaps.

In [12]:
# ── STAGE 4c: RESEARCHER ─────────────────────────────────────────────────────
print("═" * 60)
print("STAGE 4c — RESEARCHER")
print("═" * 60)
brief = blackboard["creative_brief"]
creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][0][
    "raw_response"
]
# Guard: catch missing Critic output before proceeding
critic_feedback_list = blackboard["ideation_cycles"][-1]["critic_feedback"]
if not critic_feedback_list:
    raise RuntimeError(
        "critic_feedback is empty — Critic output missing from blackboard. "
        "Re-run the Critic cell (Stage 4b) before proceeding."
    )
critic_output = critic_feedback_list[0]["raw_response"]

# Build request section if RESEARCH_REQUESTs were found
requests_section = ""
if research_requests:
    requests_section = (
        "\n\nSPECIFIC RESEARCH REQUESTS FROM THE CREATIVE TEAM:\n"
        + "\n".join(f"- {r}" for r in research_requests)
        + "\n\nAddress each of these specific requests directly before making any "
        "autonomous contribution."
    )

research_task = (
    f"You are supporting a creative studio session.\n\n"
    f"THE BRIEF:\n"
    f"Challenge: {brief['challenge']}\n"
    f"Context: {brief['context']}\n"
    f"Desired result: {brief['desired_result']}\n"
    f"{requests_section}\n\n"
    f"THE CREATIVE TEAM HAS PRODUCED:\n\n"
    f"CREATOR'S DIRECTIONS:\n{creator_output}\n\n"
    f"CRITIC'S EVALUATION:\n{critic_output}\n\n"
    f"Your task: source and cite specific external precedents, examples, or factual "
    f"findings that ground or expand what the Creative Team has produced.\n\n"
    f"For each finding:\n"
    f"TOPIC: [what area, domain, or precedent]\n"
    f"SOURCE: [name the specific source, study, institution, or example — be precise.\n"
    f"         Not 'research suggests' but the actual named source.]\n"
    f"FINDING: [what it shows]\n"
    f"RELEVANCE: [why this matters specifically for these directions]\n\n"
    f"Address specific requests first. Then make one autonomous contribution if you "
    f"have genuinely relevant external knowledge not already covered.\n\n"
    f"If the Creative Team's work is already well-grounded and no external knowledge "
    f"would add to it, say only:\n"
    f"RESEARCHER: No external contribution warranted for this cycle.\n\n"
    f"You are a sourcing agent. Do not evaluate the Creative Team's directions."
)

research_response = call_role(
    role="researcher",
    user_message=research_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=3000,
    trait_profile=trait_profiles["researcher"],
    brief_doc=read_brief_doc(session_id),
    provider=PRIMARY_PROVIDER,
)

research_entry = {
    "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
    "initiated_by": "creative_team",
    "query": "Targeted requests + autonomous contribution post-ideation",
    "sources": ["researcher_knowledge"],
    "summary": research_response,
    "influence_on_ideation": "Delivered to Creative Team for Pass 2 refinement",
}
blackboard["research_trace"].append(research_entry)
brief["research_insights"].append(research_response)
update_brief_doc(session_id, "RESEARCHER", "RESEARCH FINDINGS", research_response)
scribe_log(
    blackboard,
    "RESEARCHER",
    "research_delivered",
    f"Research delivered. Entry R{len(blackboard['research_trace']):03d} logged.",
)

print("── RESEARCHER FINDINGS ─────────────────────────────────────")
print(research_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Research logged")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4c — RESEARCHER
════════════════════════════════════════════════════════════
── RESEARCHER FINDINGS ─────────────────────────────────────
CITED FINDING  
TOPIC: Reducing the cost of a “dream” move while preserving place-feel  
SOURCE: Mihaly Csikszentmihalyi and Rochberg-Halton, *The Meaning of Things* (1981)  
FINDING: The book distinguishes between possessions and environments that support identity, autonomy, and daily meaning versus those that merely satisfy practical need. In their interviews, people often described meaningful places in terms of continuity, control, and personal significance rather than price or size alone.  
RELEVANCE: This supports A1’s question about which redesign variables preserve the emotional core. It suggests the core to protect is not “the dream address” itself but the features that stabilize identity and daily meaning: continuity, agency, and personally significant surroundings. Cost-cutt

---
## Cell 7b — Stage 4d: Creator Refinement (Pass 2)

Creator reads the Critic's evaluation and the Researcher's findings.
Refines each of the three directions — does not replace them.
Incorporates research where relevant, addresses Critic's refinement paths.

In [13]:
# ── STAGE 4d: CREATOR — REFINEMENT PASS 2 ─────────────────────────────

print("═" * 60)
print("STAGE 4d — CREATOR: REFINEMENT (PASS 2)")
print("═" * 60)

creator_refinement_task = (
    f"You are the Creator. The Critic has evaluated your directions and the "
    f"Researcher has delivered findings. Now refine your work.\n\n"
    f"YOUR ORIGINAL DIRECTIONS:\n{creator_output}\n\n"
    f"CRITIC'S EVALUATION:\n{critic_output}\n\n"
    f"RESEARCHER'S FINDINGS:\n{research_response}\n\n"
    f"Refine each of your three directions. Do not replace them — refine them.\n"
    f"For each direction:\n"
    f"- Incorporate what the research adds, where it is genuinely relevant\n"
    f"- Address the Critic's refinement path\n"
    f"- Sharpen the concept based on what you now know\n\n"
    f"Use the same format as before:\n"
    f"IDEA_ID: [A1, A2, A3 — same IDs]\n"
    f"BAND: [same band — the territory doesn't change]\n"
    f"PROBABILITY: [may adjust within band if refinement shifted the direction]\n"
    f"TITLE: [may refine]\n"
    f"CONCEPT: [refined — 2-3 sentences]\n"
    f"RATIONALE: [updated to incorporate research and critique]\n"
    f"EMOTIONAL TERRITORY: [same or refined]\n\n"
    f"If a direction is already strong and the research adds nothing to it, "
    f"say so in one sentence and keep it as written. Refinement is not mandatory "
    f"where the direction already holds."
)

creator_refined_response = call_role(
    role="creator",
    user_message=creator_refinement_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=SESSION_MODEL,
    max_tokens=2400,  # raised from 1200 — 3-direction Pass 2 needs room
    trait_profile=trait_profiles["creator"],
    brief_doc=read_brief_doc(session_id),
    provider=PRIMARY_PROVIDER,
)

# Store as cycle 2 (research-integrated refinement)
cycle_2 = {
    "cycle_number": 2,
    "creator_proposals": [{"raw_response": creator_refined_response}],
    "critic_feedback": [],
    "synthesized_directions": [],
}
blackboard["ideation_cycles"].append(cycle_2)
update_brief_doc(
    session_id, "CREATOR", "DIRECTIONS — PASS 2 (REFINED)", creator_refined_response
)
scribe_log(
    blackboard,
    "CREATOR",
    "refinement_complete",
    "Pass 2 refinement complete. Research and critique incorporated.",
)
validate_stage(blackboard, "ideation")

# Update creator_output to point to refined version for downstream cells
creator_output = creator_refined_response

print("── CREATOR REFINED DIRECTIONS ──────────────────────────────────")
print(creator_refined_response)
print("────────────────────────────────────────────────────────────────")
print(f"\n✓ Creator pass 2 complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4d — CREATOR: REFINEMENT (PASS 2)
════════════════════════════════════════════════════════════
── CREATOR REFINED DIRECTIONS ──────────────────────────────────
IDEA_ID: A1  
BAND: HIGH  
PROBABILITY: 0.71  
TITLE: Make the “right” place affordable without losing what makes it alive  
CONCEPT: Keep the emotionally right place as the anchor, but first identify the specific quality that makes it feel alive — such as a particular social network, walkable context, aesthetic texture, or sense of agency. Then stress-test only the cost-reduction moves that preserve that core: smaller unit, roommate, different tenancy type, timing shift, or a nearby sublet that keeps the same ritual/social ecology intact.  
RATIONALE: The research sharpens this direction by showing that the felt value of a place is often tied less to the address itself than to continuity, control, social ties, routine, and symbolic meaning. So the key question i

---
## Cell 7c — Stage 4e: Critic Final Evaluation + Synthesis (Pass 2)

Critic evaluates the refined directions and produces one synthesis direction.
This is the Creative Team's final submission to the Director.
The synthesis direction may become a fourth candidate in the Director's review.

In [14]:
# ── CELL 7c — STAGE 4e: CRITIC — FINAL EVALUATION + SYNTHESIS (PASS 2) (Hybrid)
# Change from v3_1: call_role() passes provider=CRITIC_PROVIDER, model=CRITIC_MODEL.
# Everything else is identical to the v3_1 cell.

print("═" * 60)
print("STAGE 4e — CRITIC: FINAL EVALUATION + SYNTHESIS (PASS 2)")
print(f"  Critic provider: {CRITIC_PROVIDER} / {CRITIC_MODEL}")
print("═" * 60)

creator_output_p2 = blackboard["ideation_cycles"][-1]["creator_proposals"][-1][
    "raw_response"
]

critic_task_p2 = f"""You are the Critic. The Creator has refined their three directions
based on your Pass 1 feedback and the Researcher's findings.

BRIEF CHALLENGE: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

REFINED CREATOR DIRECTIONS:
{creator_output_p2}

Your tasks:
1. Evaluate each refined direction concisely:

IDEA_ID: [A1, A2, A3]
ASSESSMENT: [1-2 sentences — does the refinement hold? What remains?]
REMAINING CONCERN: [one sentence — the single most important unresolved issue, if any]

2. Produce one synthesis direction that combines the strongest elements:

SYNTHESIS_ID: S1
TITLE: [short name]
CONCEPT: [2-3 sentences — what this integrated direction does]
ORIGIN_IDEAS: [which directions contributed]
WHY THIS: [one sentence — why the synthesis is stronger than any single direction]

Present all directions with trade-offs stated factually. Do not rank or recommend.
The Creative Team submits these directions to the Director for review."""

critic_response_p2 = call_role(
    role="critic",
    user_message=critic_task_p2,
    context=brief["challenge"],
    blackboard=blackboard,
    model=CRITIC_MODEL,
    max_tokens=2400,
    trait_profile=trait_profiles["critic"],
    brief_doc=read_brief_doc(session_id),
    provider=CRITIC_PROVIDER,  # ← hybrid change
)

blackboard["ideation_cycles"][-1]["critic_feedback"].append(
    {"raw_response": critic_response_p2}
)
update_brief_doc(
    session_id, "CRITIC", "FINAL EVALUATION + SYNTHESIS — PASS 2", critic_response_p2
)
scribe_log(
    blackboard,
    "CRITIC",
    "synthesis_complete",
    f"Pass 2 final evaluation and synthesis complete [{CRITIC_PROVIDER}]. "
    "Ready for Director review.",
)

print("── CRITIC FINAL EVALUATION + SYNTHESIS ─────────────────────")
print(critic_response_p2)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Critic pass 2 complete [{CRITIC_PROVIDER}]")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 4e — CRITIC: FINAL EVALUATION + SYNTHESIS (PASS 2)
  Critic provider: anthropic / claude-haiku-4-5-20251001
════════════════════════════════════════════════════════════
── CRITIC FINAL EVALUATION + SYNTHESIS ─────────────────────
# PASS 2 EVALUATION — CRITIC
*Version 1.3 — Two-Pass Assessment and Synthesis*

---

## INDIVIDUAL DIRECTION ASSESSMENTS

**IDEA_ID: A1**  
**ASSESSMENT:** The refinement holds solidly. By explicitly naming place-attachment variables (social ties, continuity, agency, ritual ecology) and testing cost-reduction moves against them rather than against cost alone, this direction transforms the question from "how cheap can we go?" to "which trade-offs preserve what makes this place alive?" The research grounding strengthens the logic considerably. The direction now has diagnostic power.

**REMAINING CONCERN:** The direction assumes the emotionally right place can be made financially viable through re

---
## Cell 8 — Stage 5 & 6: Candidate Set + Director Review

Director assembles candidate directions from the ideation cycle,
evaluates them against the brief, and decides whether to present
or send back for another round.

In [15]:
# ── STAGE 5: CANDIDATE DIRECTIONS ────────────────────────────────────────────
# ── STAGE 6: DIRECTOR REVIEW ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 5 — CANDIDATE DIRECTIONS")
print("STAGE 6 — DIRECTOR REVIEW")
print("═" * 60)

# Read from the latest ideation cycle (cycle 2 — refined)
creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][-1][
    "raw_response"
]
critic_output = blackboard["ideation_cycles"][-1]["critic_feedback"][-1]["raw_response"]

review_task = f"""You are reviewing the Creative Team\'s final submission before presenting to the PIL.
The team has completed two passes: initial ideation, research integration, and refinement.

BRIEF:
Challenge: {brief['challenge']}
Desired result: {brief['desired_result']}
Constraints: {", ".join(brief['constraints'])}

REFINED CREATOR DIRECTIONS:
{creator_output[:900]}

CRITIC FINAL EVALUATION + SYNTHESIS:
{critic_output[:900]}

Your task:
1. Select 3 directions to present — from the refined set plus the Critic\'s synthesis (S1).
   Include S1 if it is stronger than any of the three refined directions.
2. Evaluate the candidate set.
3. Decide: PRESENT or ITERATE.

Return ONLY a JSON object — no preamble, no markdown fences:

{{
  "candidates": [
    {{"id": "C1", "title": "exact direction name", "type": "evolutionary|evolutionary_variant|conceptual_leap|sacrificial_probe", "source_idea_id": "A1"}},
    {{"id": "C2", "title": "...", "type": "...", "source_idea_id": "A2"}},
    {{"id": "C3", "title": "...", "type": "...", "source_idea_id": "S1"}}
  ],
  "alignment": "one sentence",
  "distinctiveness": "one sentence",
  "balance": "one sentence",
  "clarity": "one sentence",
  "decision": "PRESENT",
  "reason": "one sentence"
}}

Set decision to "ITERATE" with reason if the team\'s work does not meet standard."""

review_response = call_role(
    role="director",
    user_message=review_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_FAST_MODEL,
    max_tokens=800,
    brief_doc=read_brief_doc(session_id),
    provider=PRIMARY_PROVIDER,
)

# ── Parse Director review — JSON with validation gate ─────────────────────────
review = blackboard["director_review"]

try:
    clean = review_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    parsed = json.loads(clean.strip())

    review["alignment_with_brief"] = parsed.get("alignment", "")
    review["distinctiveness_assessment"] = parsed.get("distinctiveness", "")
    review["balance_assessment"] = parsed.get("balance", "")
    review["clarity_assessment"] = parsed.get("clarity", "")
    decision = parsed.get("decision", "PRESENT")
    review["iteration_required"] = "ITERATE" in decision.upper()

    candidates = parsed.get("candidates", [])
    if not candidates:
        raise ValueError("Director review returned no candidates")

    for c in candidates:
        blackboard["candidate_set"].append(
            {
                "direction_id": c.get(
                    "id", f"D{len(blackboard['candidate_set'])+1:03d}"
                ),
                "internal_type": c.get("type", "evolutionary").lower(),
                "description": c.get("title", ""),
                "reasoning_summary": "",
                "research_influences": [],
                "critic_assessment": {
                    "strengths": [],
                    "concerns": [],
                    "refinement_notes": [],
                },
            }
        )

    print("✓ Director review parsed successfully")
    print(f"  Decision   : {decision}")
    print(f"  Candidates : {len(candidates)}")

except (json.JSONDecodeError, ValueError) as e:
    print(f"✗ Director review parsing failed: {e}")
    print(review_response)
    raise RuntimeError("Director review parsing failed — candidate set empty.")

update_brief_doc(session_id, "DIRECTOR", "DIRECTOR REVIEW", review_response)
scribe_log(
    blackboard,
    "DIRECTOR",
    "review_complete",
    f"Director review complete. Decision: {'ITERATE' if review['iteration_required'] else 'PRESENT'}. "
    f"{len(blackboard['candidate_set'])} candidates selected.",
)

if not review["iteration_required"]:
    validate_stage(blackboard, "candidate_set")

print("── DIRECTOR REVIEW ─────────────────────────────────────────")
print(review_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Director review complete")
print(f"  Candidates selected : {len(blackboard['candidate_set'])}")
print(f"  Iteration required  : {review['iteration_required']}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 5 — CANDIDATE DIRECTIONS
STAGE 6 — DIRECTOR REVIEW
════════════════════════════════════════════════════════════
✓ Director review parsed successfully
  Decision   : PRESENT
  Candidates : 3
── DIRECTOR REVIEW ─────────────────────────────────────────
{"candidates":[{"id":"C1","title":"Make the “right” place affordable without losing what makes it alive","type":"evolutionary","source_idea_id":"A1"},{"id":"C2","title":"Choose a time-bounded life-support structure, not a final location","type":"conceptual_leap","source_idea_id":"A2"},{"id":"C3","title":"The Three-Layer Decision Framework: Portability → Place-Fit → Cost-Viability","type":"conceptual_leap","source_idea_id":"S1"}],"alignment":"The set stays tightly aligned to the brief by testing livability, financial strain, and hidden conditions rather than collapsing the problem into a false either/or.","distinctiveness":"The three directions are meaningfully different: on

---
## Cell 8a — Stage 6a: Iteration Loop

Fires only when the Director signals ITERATE.
Re-runs the full Creative Team: Creator → Critic → Researcher.
Then re-runs Director review to produce a new candidate set.
Cell 9 (Presentation) proceeds normally from the updated candidate_set.

If ITERATE is not required, this cell prints a confirmation and passes through.

In [16]:
# ── STAGE 6a: ITERATION LOOP ─────────────────────────────────────────────────

print("═" * 60)
print("STAGE 6a — ITERATION LOOP")
print("═" * 60)

if not review["iteration_required"]:
    print("✓ Director decision: PRESENT — no iteration needed")
    print(f"✓ Candidates in set: {len(blackboard['candidate_set'])}")

else:
    print("⚠ Director decision: ITERATE — re-running full Creative Team")
    print()

    # Clear existing ideation cycle and candidate set for a clean cycle 2
    blackboard["ideation_cycles"] = []
    blackboard["candidate_set"] = []

    # ── CREATOR — CYCLE 2 ─────────────────────────────────────────────────────
    print("── CREATOR: CYCLE 2 ────────────────────────────────────────")

    iteration_context = (
        f"The Director reviewed the first ideation cycle and requested a full re-run.\n\n"
        f"DIRECTOR'S FEEDBACK:\n{review_response}\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Desired result: {brief['desired_result']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        f"Generate 4 new distinct directions that address the Director's specific concerns above.\n"
        f"Use Verbalized Sampling: include probability scores. Include at least one below 0.15.\n\n"
        f"For each direction:\n"
        f"IDEA_ID: [A1, A2, A3, A4]\n"
        f"TITLE: [short evocative name]\n"
        f"PROBABILITY: [0.0-1.0]\n"
        f"CONCEPT: [2-3 sentences]\n"
        f"RATIONALE: [why this is worth exploring]\n"
        f"EMOTIONAL TERRITORY: [what feeling does this occupy?]"
    )

    creator_response_iter = call_role(  # ← FIXED: was role="critic"
        role="creator",
        user_message=iteration_context,  # ← FIXED: was critic_task_iter (undefined here)
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,  # ← FIXED: was CRITIC_MODEL
        max_tokens=1800,
        trait_profile=trait_profiles["creator"],
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,  # ← FIXED: was CRITIC_PROVIDER
    )

    cycle_2 = {
        "cycle_number": 2,
        "creator_proposals": [{"raw_response": creator_response_iter}],
        "critic_feedback": [],
        "synthesized_directions": [],
    }
    blackboard["ideation_cycles"].append(cycle_2)
    update_brief_doc(
        session_id, "CREATOR", "DIRECTIONS — CYCLE 2", creator_response_iter
    )
    scribe_log(
        blackboard,
        "CREATOR",
        "iteration_ideation_complete",
        "Cycle 2 ideation complete following Director iteration request.",
    )
    validate_stage(blackboard, "ideation")

    print(creator_response_iter)
    print()

    # ── CRITIC — CYCLE 2 ──────────────────────────────────────────────────────
    print("── CRITIC: CYCLE 2 ─────────────────────────────────────────")

    critic_task_iter = (
        f"The Creator has proposed a new set of directions following Director feedback.\n\n"
        f"BRIEF CHALLENGE: {brief['challenge']}\n"
        f"DESIRED RESULT: {brief['desired_result']}\n\n"
        f"CREATOR'S NEW DIRECTIONS:\n{creator_response_iter}\n\n"
        f"Evaluate each direction. For each:\n"
        f"IDEA_ID: [matching the Creator's ID]\n"
        f"STRENGTHS: [what genuinely works]\n"
        f"WEAKNESSES: [what is underdeveloped or misaligned]\n"
        f"REFINEMENT: [one concrete suggestion]\n\n"
        f"Then propose ONE synthesis direction:\n"
        f"SYNTHESIS_ID: S1\n"
        f"TITLE: [name]\n"
        f"CONCEPT: [2-3 sentences]\n"
        f"ORIGIN_IDEAS: [which Creator ideas it draws from]\n\n"
        f"Be analytically rigorous."
    )

    critic_response_iter = call_role(
        role="critic",
        user_message=critic_task_iter,
        context=brief["challenge"],
        blackboard=blackboard,
        model=CRITIC_MODEL,  # ← FIXED: was SESSION_MODEL
        max_tokens=1800,
        trait_profile=trait_profiles["critic"],
        brief_doc=read_brief_doc(session_id),
        provider=CRITIC_PROVIDER,
    )

    blackboard["ideation_cycles"][-1]["critic_feedback"].append(
        {"raw_response": critic_response_iter}
    )
    update_brief_doc(session_id, "CRITIC", "CRITIQUE — CYCLE 2", critic_response_iter)
    scribe_log(
        blackboard,
        "CRITIC",
        "iteration_evaluation_complete",
        "Cycle 2 Critic evaluation complete.",
    )
    validate_stage(blackboard, "critic")

    print(critic_response_iter)
    print()

    # ── RESEARCHER — CYCLE 2 ──────────────────────────────────────────────────
    print("── RESEARCHER: CYCLE 2 ─────────────────────────────────────")

    research_task_iter = (
        f"You are supporting a creative studio session. The Creative Team has completed "
        f"a second ideation cycle.\n\n"
        f"THE BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Context: {brief['context']}\n"
        f"Desired result: {brief['desired_result']}\n\n"
        f"CREATOR'S DIRECTIONS (CYCLE 2):\n{creator_response_iter}\n\n"
        f"CRITIC'S EVALUATION (CYCLE 2):\n{critic_response_iter}\n\n"
        f"Your task: source and cite 2-3 external precedents, examples, or factual findings\n"
        f"from outside this problem's domain that could ground or expand what the Creative\n"
        f"Team has produced. Respond to the actual directions above.\n\n"
        f"For each finding:\n"
        f"TOPIC: [what area, domain, or precedent]\n"
        f"SOURCE: [name the specific source, study, institution, or example — be precise]\n"
        f"FINDING: [what it shows]\n"
        f"RELEVANCE: [why this matters for these specific directions]\n\n"
        f"If you have no external knowledge that would genuinely add here, say only:\n"
        f"RESEARCHER: No external contribution warranted for this cycle.\n\n"
        f"You are a sourcing agent, not an analyst. Do not evaluate the Creative Team's work."
    )

    research_response_iter = call_role(
        role="researcher",
        user_message=research_task_iter,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1200,
        trait_profile=trait_profiles["researcher"],
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    research_entry_iter = {
        "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
        "initiated_by": "director_iteration",
        "query": "Post-ideation cycle 2 enrichment",
        "sources": ["researcher_knowledge"],
        "summary": research_response_iter,
        "influence_on_ideation": "Cycle 2 post-iteration research",
    }
    blackboard["research_trace"].append(research_entry_iter)
    brief["research_insights"].append(research_response_iter)
    update_brief_doc(
        session_id, "RESEARCHER", "RESEARCH — CYCLE 2", research_response_iter
    )
    scribe_log(
        blackboard,
        "RESEARCHER",
        "iteration_research_delivered",
        f"Cycle 2 research. Entry R{len(blackboard['research_trace']):03d} logged.",
    )

    print(research_response_iter)
    print()

    # ── DIRECTOR REVIEW — CYCLE 2 ─────────────────────────────────────────────
    print("── DIRECTOR REVIEW: CYCLE 2 ────────────────────────────────")

    review_task_iter = (
        f"You are reviewing the studio's second ideation cycle before presenting to the PIL.\n\n"
        f"BRIEF:\n"
        f"Challenge: {brief['challenge']}\n"
        f"Desired result: {brief['desired_result']}\n"
        f"Constraints: {', '.join(brief['constraints'])}\n\n"
        f"CREATOR DIRECTIONS (CYCLE 2):\n{creator_response_iter[:800]}\n\n"
        f"CRITIC EVALUATION (CYCLE 2):\n{critic_response_iter[:800]}\n\n"
        f"Select 3 directions to present. Return ONLY a JSON object — "
        f"no preamble, no markdown fences:\n\n"
        f"{{\n"
        f'  "candidates": [\n'
        f'    {{"id": "C1", "title": "...", "type": "evolutionary|evolutionary_variant|conceptual_leap|sacrificial_probe", "source_idea_id": "A1"}},\n'
        f'    {{"id": "C2", "title": "...", "type": "...", "source_idea_id": "A2"}},\n'
        f'    {{"id": "C3", "title": "...", "type": "...", "source_idea_id": "S1"}}\n'
        f"  ],\n"
        f'  "alignment": "...",\n'
        f'  "distinctiveness": "...",\n'
        f'  "balance": "...",\n'
        f'  "clarity": "...",\n'
        f'  "decision": "PRESENT",\n'
        f'  "reason": "..."\n'
        f"}}"
    )

    review_response_iter = call_role(
        role="director",
        user_message=review_task_iter,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_FAST_MODEL,
        max_tokens=1200,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    # Parse cycle 2 Director review
    try:
        clean_iter = review_response_iter.strip()
        if clean_iter.startswith("```"):
            clean_iter = clean_iter.split("```")[1]
            if clean_iter.startswith("json"):
                clean_iter = clean_iter[4:]
        parsed_iter = json.loads(clean_iter.strip())

        review["alignment_with_brief"] = parsed_iter.get("alignment", "")
        review["distinctiveness_assessment"] = parsed_iter.get("distinctiveness", "")
        review["balance_assessment"] = parsed_iter.get("balance", "")
        review["clarity_assessment"] = parsed_iter.get("clarity", "")
        review["iteration_required"] = False  # reset after cycle 2 review

        for c in parsed_iter.get("candidates", []):
            blackboard["candidate_set"].append(
                {
                    "direction_id": c.get(
                        "id", f"D{len(blackboard['candidate_set'])+1:03d}"
                    ),
                    "internal_type": c.get("type", "evolutionary").lower(),
                    "description": c.get("title", ""),
                    "reasoning_summary": "",
                    "research_influences": [],
                    "critic_assessment": {
                        "strengths": [],
                        "concerns": [],
                        "refinement_notes": [],
                    },
                }
            )

        print("✓ Cycle 2 Director review parsed successfully")

    except (json.JSONDecodeError, ValueError) as e:
        print(f"✗ Cycle 2 Director review parsing failed: {e}")
        print(review_response_iter)
        raise RuntimeError(
            "Iteration Director review failed — candidate set still empty.\n"
            "Check the Director prompt or model output above."
        )

    update_brief_doc(
        session_id, "DIRECTOR", "DIRECTOR REVIEW — CYCLE 2", review_response_iter
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "iteration_review_complete",
        f"Cycle 2 Director review complete. {len(blackboard['candidate_set'])} candidates selected.",
    )
    validate_stage(blackboard, "candidate_set")

    # Update creator_output and critic_output so downstream cells (9 and 11)
    # read from cycle 2, not cycle 1.
    creator_output = creator_response_iter
    critic_output = critic_response_iter

    print()
    print(f"✓ Iteration complete")
    print(f"✓ Candidates in set : {len(blackboard['candidate_set'])}")
    print(f"✓ Reasoning trace   : {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 6a — ITERATION LOOP
════════════════════════════════════════════════════════════
✓ Director decision: PRESENT — no iteration needed
✓ Candidates in set: 3


---
## Cell 9 — Stage 7: Presentation

Director presents the candidate directions to the PIL.
Each direction includes reasoning. Tone is clear, professional, non-salesy.

In [17]:
# ── STAGE 7: PRESENTATION ────────────────────────────────────────────────────
print("═" * 60)
print("STAGE 7 — PRESENTATION")
print("═" * 60)

# ── Pull all studio outputs the Director needs to write from ──────────────────
creator_output = blackboard["ideation_cycles"][-1]["creator_proposals"][-1][
    "raw_response"
]
critic_output = blackboard["ideation_cycles"][-1]["critic_feedback"][-1]["raw_response"]

# Researcher output — Director needs this to write Research Foundation sections
# that are legible to the PIL, who has never seen the research
researcher_output = (
    blackboard["research_trace"][-1]["summary"]
    if blackboard["research_trace"]
    else "No research conducted this session."
)

# ── Build candidate summary with full identifying info ────────────────────────
candidate_summary = "\n".join(
    [
        f"- ID: {c['direction_id']} | Title: {c.get('title', c['description'][:60])} "
        f"| Type: {c['internal_type']} | Source: {c.get('source_idea_id', 'unknown')}"
        for c in blackboard["candidate_set"]
    ]
)

# ── Presentation task — aligned to director.md v1.9 PRESENTATION section ─────
presentation_task = f"""You are presenting the studio's work to the PIL.
Three directions are ready. This is the deliverable the PIL will read and
make decisions from. It must do full justice to the work the studio produced.

BRIEF CHALLENGE: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

CANDIDATE DIRECTIONS:
{candidate_summary}

FULL STUDIO OUTPUT TO DRAW FROM:

CREATOR DIRECTIONS:
{creator_output[:2500]}

CRITIC EVALUATION:
{critic_output[:2500]}

RESEARCHER FINDINGS:
{researcher_output[:2500]}

---

For each direction, use this exact format with these exact section labels:

**[Direction Title]**
A meaningful name that conveys the mechanism — not a marketing label.
The title should tell the PIL what the direction actually does.

**The Core Move**
Eight to twelve sentences. Explain what this direction does, how it works
mechanically, the main tactical details, and what specific challenge it solves
for this person's problem. Develop the idea fully — not just what it is, but
how it operates in practice, what it changes, and why this particular approach
addresses the underlying tension the brief identified. Ground it in what research
and discovery revealed, not abstract principles. The PIL should hear their own
problem in the description and understand the direction well enough to picture it
in action and begin evaluating whether it fits their situation. This is the heart
of the proposal, where the PIL will start to intuitively feel if it works.
If the mechanism, the rationale, and the human context are not all present,
the section is not complete.

**The Innovation**
One sentence only: what the studio found that the PIL could not have arrived at
alone. This is where research insight, creative tension, or an unexpected reframe
gets surfaced. It is the sentence that earns the studio's existence. If it could
apply to any brief, rewrite it.

**What It Asks**
One sentence: what this direction honestly requires — of the PIL, their
stakeholders, or the situation. A direction that seems to cost nothing is either
obvious or incomplete.

**Research Foundation**
Two to four sentences. THE PIL HAS NOT SEEN THE RESEARCH — write as if explaining
it to someone encountering it for the first time. Name the domain, the key
finding, and why it is relevant to this specific direction.
If you reference a named case, city, study, or data point, give one sentence
of context: what happened, what it showed, and why it applies here.
A reference without context is not a research foundation — it is a citation
the PIL cannot evaluate. The substance must be legible without background knowledge.

**Invitation to Go Deeper**
One sentence, specific — not a vague offer. Name the domain of the research or
the precise evaluative concern. Two forms:
- Research anchor: "This direction draws on [named domain] — ask me for the full
  findings if you want to weigh the evidence before deciding."
- Evaluation signal: "The evaluation identified [specific concern] as the main
  risk — ask me for the refinement path if you want to weigh it before deciding."

---

Sequence the directions intentionally — lead with the strongest, end with
the most unexpected or exploratory. If one direction is clearly superior,
express a subtle preference. If they are equally viable, remain neutral.

Close with a single question that invites the PIL's reaction.

Tone: warm, direct, confident. No hype. Never use 'exciting', 'powerful',
'innovative', 'robust', or hollow affirmations."""

presentation_response = call_role(
    role="director",
    user_message=presentation_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=3500,
    brief_doc=read_brief_doc(session_id),
    provider=PRIMARY_PROVIDER,
)

# ── Store presentation to blackboard ─────────────────────────────────────────
blackboard["presentation"]["ordered_directions"] = [
    c["direction_id"] for c in blackboard["candidate_set"]
]
blackboard["presentation"]["director_commentary"].append(presentation_response)

scribe_log(
    blackboard,
    "DIRECTOR",
    "presentation_delivered",
    f"Presentation delivered to PIL. {len(blackboard['candidate_set'])} directions presented.",
)

print("── DIRECTOR PRESENTS TO PIL ────────────────────────────────")
print(presentation_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Presentation complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 7 — PRESENTATION
════════════════════════════════════════════════════════════
── DIRECTOR PRESENTS TO PIL ────────────────────────────────
Here are the three directions I’d put in front of you. I’m leading with the strongest one because it does the best job of turning this from a vague emotional dilemma into a usable decision structure.

**The Three-Layer Decision Framework: Portability → Place-Fit → Cost-Viability**

**The Core Move**  
This direction says you should not start by choosing between the two places. You should start by identifying what actually makes a place livable for you in this season: the concrete practices, rhythms, and conditions that let your life feel intact rather than performative or depleted. That might mean solitude, a certain kind of neighborhood movement, proximity to particular people, aesthetic texture, distance from distraction, or enough control over your space to feel like yourself. Onc

---
## Cell 10 — Stage 8: User Reaction

PIL responds to the candidate directions.
Director extracts creative signals from the reaction.
These signals determine whether to proceed to synthesis or trigger a second loop.

In [18]:
# ── CELL 10 — STAGE 8: USER REACTION ─────────────────────────────────────────

print("═" * 60)
print("STAGE 8 — USER REACTION")
print("═" * 60)

pil_reaction = input("Your reaction: ").strip()
print()

# Director extracts signals — JSON format prevents markdown parsing failures
signal_task = f"""The PIL responded to the presentation:
"{pil_reaction}"

Extract the creative signals from this reaction.

Return ONLY a JSON object — no preamble, no markdown fences:

{{
  "selected_direction": "<which direction resonated, e.g. C1, C2, C3, or none>",
  "signal_1": "<key preference or instinct revealed>",
  "signal_2": "<what they want more or less of>",
  "signal_3": "<any new insight or direction revealed, or empty string>",
  "second_loop": <true if the reaction reveals new information that would meaningfully change the creative directions, false if the reaction confirms an existing direction>,
  "second_loop_reason": "<why second loop is or isn't needed — one sentence>"
}}

second_loop should be true only if the PIL's reaction reveals something genuinely new
that the Creative Team did not have when they built the directions — a reframe, a
constraint, a signal that substantially changes what the synthesis should be.
It should NOT trigger just because the PIL wants to push a direction further."""

signal_response = call_role(
    role="director",
    user_message=signal_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_FAST_MODEL,
    max_tokens=800,
    brief_doc=read_brief_doc(session_id),
    provider=PRIMARY_PROVIDER,
)

# Parse JSON response — with fallback if Director returns narrative instead
try:
    clean = signal_response.strip()
    if clean.startswith("```"):
        clean = clean.split("```")[1]
        if clean.startswith("json"):
            clean = clean[4:]
    signal_data = json.loads(clean.strip())
    second_loop_needed = bool(signal_data.get("second_loop", False))
    second_loop_reason = signal_data.get("second_loop_reason", "")
    signals_list = [
        signal_data.get("signal_1", ""),
        signal_data.get("signal_2", ""),
        signal_data.get("signal_3", ""),
    ]
    signals_list = [s for s in signals_list if s.strip()]
    selected_dir = signal_data.get("selected_direction", "")
except (json.JSONDecodeError, ValueError, AttributeError):
    second_loop_needed = False
    second_loop_reason = "JSON parse failed — narrative response stored"
    signals_list = []
    selected_dir = ""
    print("  ⚠ Signal extraction returned non-JSON — stored as narrative")

# Store user response
user_resp = blackboard["user_response"]
user_resp["feedback_summary"] = pil_reaction
user_resp["signals_extracted"].append(signal_response)
if selected_dir:
    user_resp["selected_direction"] = selected_dir

# Trigger second loop if needed
blackboard["second_exploration"]["triggered"] = second_loop_needed
if second_loop_needed:
    blackboard["second_exploration"]["reason"] = second_loop_reason

# ── Director acknowledges the PIL's selection ─────────────────────────────────
# This is the missing step: the PIL must hear that they were understood
# before Cell 10b fires. Without this, the depth question arrives with
# no confirmation and feels disconnected or like the selection was ignored.

_direction_name = ""
for c in blackboard["candidate_set"]:
    if c["direction_id"] == selected_dir:
        _direction_name = c["description"]
        break

acknowledgment_task = (
    f"The PIL has reacted to the presentation and selected a direction.\n\n"
    f"PIL REACTION: {pil_reaction}\n\n"
    f"SELECTED DIRECTION: {selected_dir or 'unclear'}"
    + (f" — {_direction_name}" if _direction_name else "")
    + f"\n\nWrite ONE sentence acknowledging what they selected and that you heard them. "
    f"Warm and direct. Do not repeat the direction title back verbatim. "
    f"Do not ask a question. Do not move to synthesis yet. Just confirm you heard them."
)

acknowledgment = call_role(
    role="director",
    user_message=acknowledgment_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_FAST_MODEL,
    max_tokens=80,
    brief_doc=read_brief_doc(session_id),
    provider=PRIMARY_PROVIDER,
)

print("── DIRECTOR ────────────────────────────────────────────────")
print(acknowledgment)
print("────────────────────────────────────────────────────────────")
print()

# Update Studio Brief Document
update_brief_doc(
    session_id,
    "DIRECTOR",
    "PIL SIGNALS",
    f"**PIL reaction:** {pil_reaction}\n\n"
    f"**Selected:** {selected_dir}\n\n"
    f"**Director acknowledged:** {acknowledgment}\n\n"
    f"**Signals extracted:**\n{signal_response}\n\n"
    f"**Second loop:** {second_loop_needed}"
    + (f" — {second_loop_reason}" if second_loop_reason else ""),
)

scribe_log(
    blackboard,
    "DIRECTOR",
    "signals_extracted",
    f"PIL reaction processed. Second loop: {second_loop_needed}. "
    f"Key signal: '{pil_reaction[:60]}...'",
)

print(f"✓ Signals extracted")
print(f"  Selected direction  : {selected_dir or 'not parsed'}")
print(f"  Second loop         : {second_loop_needed}")
if second_loop_needed:
    print(f"  Reason              : {second_loop_reason}")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 8 — USER REACTION
════════════════════════════════════════════════════════════

── DIRECTOR ────────────────────────────────────────────────
Got it — you’re choosing the structured path that tests what must be true in order, and I hear that clearly.
────────────────────────────────────────────────────────────

✓ Signals extracted
  Selected direction  : C3
  Second loop         : False
✓ Reasoning trace: 38 entries


---
## Cell 10b — Stage 8b: Depth Extraction

Director uses one extraction technique from the toolkit to probe deeper
before committing to final synthesis.
Director asks one focused question, PIL responds via live input.

In [19]:
# ── CELL 10b — STAGE 8b: DEPTH EXTRACTION ────────────────────────────────────
#
# Director probes one level deeper before committing to synthesis.
# SIGNAL QUALITY GATE: if Cell 10 already captured a clear selection
# and sufficient signal, this cell passes through without asking anything.
# Maximum one follow-up question, maximum one re-probe if response is thin.

print("═" * 60)
print("STAGE 8b — DEPTH EXTRACTION")
print("═" * 60)

# ── Signal quality gate ───────────────────────────────────────────────────────
# If the PIL's reaction in Cell 10 was clear — a direction was selected and
# at least one substantive signal was extracted — skip the depth question.
# The PIL has already given what the Director needs.

_reaction_words = len(pil_reaction.split())
_has_selection = bool(selected_dir and selected_dir not in ("none", ""))
_has_signals = len(signals_list) >= 1
_reaction_is_rich = _reaction_words >= 12  # more than a one-liner

_sufficient_signal = _has_selection and _has_signals and _reaction_is_rich

if _sufficient_signal:
    print("✓ Signal sufficient — depth extraction not needed")
    print(f"  Selection clear     : {selected_dir}")
    print(f"  Signals captured    : {len(signals_list)}")
    print(f"  Reaction length     : {_reaction_words} words")
    print()

    # Store pass-through so downstream cells have the variable they need
    depth_response = pil_reaction
    scribe_log(
        blackboard,
        "DIRECTOR",
        "depth_extraction_complete",
        f"Stage 8b passed through — signal sufficient. "
        f"Selection: {selected_dir}. Words: {_reaction_words}.",
    )
    blackboard["user_response"]["signals_extracted"].append(
        f"DEPTH SIGNAL (Stage 8b): [passed through — reaction was sufficient]"
    )
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "DEPTH SIGNAL",
        f"**Passed through** — PIL reaction was clear and sufficient.\n\n"
        f"**Selection:** {selected_dir}\n"
        f"**PIL reaction (carried forward):** {pil_reaction}",
    )

else:
    # Signal was thin or selection was unclear — probe one level deeper
    depth_task = (
        f"The PIL has reacted to the presentation. Before producing the final synthesis,\n"
        f"probe one level deeper using a single extraction technique from your toolkit.\n\n"
        f"PIL REACTION:\n{pil_reaction}\n\n"
        f"SIGNALS EXTRACTED SO FAR:\n{signal_response}\n\n"
        f"Choose ONE technique that would surface the most useful additional signal right now.\n"
        f"Ask a single focused question — not multiple questions.\n"
        f"Keep it conversational. Do not explain why you're asking. Just ask."
    )

    depth_question = call_role(
        role="director",
        user_message=depth_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_FAST_MODEL,
        max_tokens=150,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    print("── DIRECTOR ────────────────────────────────────────────────")
    print(depth_question)
    print("────────────────────────────────────────────────────────────")
    print()
    depth_response = input("> ").strip()

    # -- Minimum signal threshold ---------------------------------------------
    # A response of ≤10 words adds little. One re-probe with a different
    # technique before logging. Maximum one re-probe.

    _thin_signal = len(depth_response.split()) <= 10

    if _thin_signal:
        print()
        print("  (Director probing further...)")
        print()

        reprobe_task = (
            f"The PIL gave a brief or noncommittal response to your depth question.\n\n"
            f"YOUR QUESTION: {depth_question}\n"
            f"PIL RESPONSE: {depth_response}\n\n"
            f"Try a different extraction technique — one that approaches from a different "
            f"angle and is more likely to open something up. Do not repeat the same question "
            f"or the same technique. One question only. Keep it light and genuinely curious."
        )

        reprobe_question = call_role(
            role="director",
            user_message=reprobe_task,
            context=brief["challenge"],
            blackboard=blackboard,
            model=DIRECTOR_FAST_MODEL,
            max_tokens=150,
            brief_doc=read_brief_doc(session_id),
            provider=PRIMARY_PROVIDER,
        )

        print("── DIRECTOR ────────────────────────────────────────────────")
        print(reprobe_question)
        print("────────────────────────────────────────────────────────────")
        print()
        reprobe_response = input("> ").strip()

        if len(reprobe_response.split()) > len(depth_response.split()):
            depth_response = reprobe_response
            print("  (Using follow-up response — more signal)")
        else:
            print("  (Using original response — follow-up did not add signal)")

    # Store depth signal
    blackboard["user_response"]["signals_extracted"].append(
        f"DEPTH SIGNAL (Stage 8b): {depth_response}"
    )
    update_brief_doc(
        session_id,
        "DIRECTOR",
        "DEPTH SIGNAL",
        f"**Director question:** {depth_question}\n\n"
        f"**PIL response:** {depth_response}"
        + (
            f"\n\n**Note:** Thin signal detected — re-probe attempted."
            if _thin_signal
            else ""
        ),
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "depth_extraction_complete",
        f"Stage 8b depth extraction complete. Signal: '{depth_response[:80]}'",
    )

print()
print("✓ Depth signal captured")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 8b — DEPTH EXTRACTION
════════════════════════════════════════════════════════════
── DIRECTOR ────────────────────────────────────────────────
Before I point you to one, what’s the one thing you cannot afford to lose in this move?
────────────────────────────────────────────────────────────


  (Director probing further...)

── DIRECTOR ────────────────────────────────────────────────
Interesting. If the move had a “keep / lose / maybe” drawer, what would you put in each one?
────────────────────────────────────────────────────────────

  (Using follow-up response — more signal)

✓ Depth signal captured
✓ Reasoning trace: 41 entries



---
## Cell 10c — Stage 8c: Second Loop
 
Fires only when the Director's signal extraction identified new information
that warrants a fresh Creative Team run.
 
If second_exploration.triggered is True: re-runs Creator → Critic → Researcher
with the PIL's reaction signals as additional brief context.
Updated directions replace the original candidate set for synthesis.
 
If False: passes through immediately. No API calls made.


In [20]:
# ── STAGE 8c: FOCUSED REFINEMENT ─────────────────────────────────────────────
#
# Fires only when second_exploration.triggered is True.
#
# Design: the second loop is a funnel, not a new exploration.
# The Director restates the brief based on PIL feedback, produces
# 1-2 targeted directions (Creator only, no Critic), conditionally
# calls the Researcher, selects 2-3 final candidates based on the
# rejection signal type, presents them to the PIL, gets a reaction,
# and logs the final signal before synthesis.
#
# If not triggered: passes through immediately.

print("═" * 60)
print("STAGE 8c — FOCUSED REFINEMENT")
print("═" * 60)

if not blackboard["second_exploration"]["triggered"]:
    print("✓ Second loop not required — proceeding to synthesis")
    print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

else:
    print("⚠ Second loop triggered — focused refinement pass\n")
    print(f"  Reason: {blackboard['second_exploration']['reason']}\n")

    all_signals = "\n\n".join(blackboard["user_response"]["signals_extracted"])

    # ── Step 1: Director restatement ─────────────────────────────────────────
    # The Director reads the PIL reaction and existing work, then:
    # (a) writes a focused brief addition naming what needs to change
    # (b) classifies the rejection signal type
    # (c) decides whether new research is needed

    print("── DIRECTOR: RESTATEMENT ────────────────────────────────────")

    # Summarize first-loop candidates for Director context
    first_loop_summary = "\n".join(
        f"- {c['direction_id']}: {c['description']}"
        for c in blackboard["candidate_set"]
    )

    restatement_task = (
        f"You are the Director. The PIL has reacted to the first set of directions "
        f"and a second creative pass has been triggered.\n\n"
        f"ORIGINAL BRIEF CHALLENGE:\n{brief['challenge']}\n\n"
        f"FIRST LOOP DIRECTIONS:\n{first_loop_summary}\n\n"
        f"PIL REACTION AND SIGNALS:\n{all_signals}\n\n"
        f"Do three things and return ONLY a JSON object — no preamble:\n\n"
        f"{{\n"
        f'  "rejection_type": "flat | partial | reframe",\n'
        f'  "surviving_direction": "direction_id of strongest first-loop direction to carry forward, or null if flat rejection",\n'
        f'  "restated_brief": "2-3 sentences: what specifically needs to change or be added based on PIL reaction",\n'
        f'  "research_needed": true or false,\n'
        f'  "research_question": "one specific question if research_needed is true, else null"\n'
        f"}}\n\n"
        f"rejection_type definitions:\n"
        f"- flat: PIL rejected all directions; nothing carries forward\n"
        f"- partial: PIL showed energy toward one direction but it didn't land fully; "
        f"that direction may carry forward alongside new work\n"
        f"- reframe: PIL didn't dislike the directions but identified a missing dimension; "
        f"new directions address what was missing\n\n"
        f"surviving_direction: only relevant for partial — the direction the PIL showed "
        f"most energy toward. Null for flat and reframe.\n\n"
        f"research_needed: true only if the restated brief requires genuinely new external "
        f"knowledge not already present in the session. The Researcher has already "
        f"contributed substantially — default to false unless the gap is clear."
    )

    restatement_response = call_role(
        role="director",
        user_message=restatement_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_FAST_MODEL,
        max_tokens=400,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    # Parse Director restatement
    try:
        clean_r = restatement_response.strip()
        if clean_r.startswith("```"):
            clean_r = clean_r.split("```")[1]
            if clean_r.startswith("json"):
                clean_r = clean_r[4:]
        restatement = json.loads(clean_r.strip())
        rejection_type = restatement.get("rejection_type", "reframe")
        surviving_direction = restatement.get("surviving_direction")
        restated_brief = restatement.get("restated_brief", "")
        research_needed = bool(restatement.get("research_needed", False))
        research_question = restatement.get("research_question")
    except (json.JSONDecodeError, ValueError):
        rejection_type = "reframe"
        surviving_direction = None
        restated_brief = blackboard["second_exploration"]["reason"]
        research_needed = False
        research_question = None
        print("  ⚠ Restatement parse failed — using trigger reason as restated brief")

    print(f"  Rejection type     : {rejection_type}")
    print(f"  Surviving direction: {surviving_direction or 'none'}")
    print(f"  Research needed    : {research_needed}")
    print(f"  Restated brief     : {restated_brief}")
    print()

    update_brief_doc(
        session_id,
        "DIRECTOR",
        "FOCUSED REFINEMENT — RESTATEMENT",
        f"**Rejection type:** {rejection_type}\n\n"
        f"**PIL signal:** {all_signals}\n\n"
        f"**Restated brief:** {restated_brief}\n\n"
        f"**Surviving direction:** {surviving_direction or 'none'}\n\n"
        f"**Research needed:** {research_needed}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "restatement_complete",
        f"Restatement complete. Rejection: {rejection_type}. "
        f"Research needed: {research_needed}.",
    )

    # ── Step 2: Conditional Researcher call ──────────────────────────────────
    focused_research = ""

    if research_needed and research_question:
        print("── RESEARCHER: FOCUSED RESEARCH ─────────────────────────────")

        focused_research_task = (
            f"You are supporting a creative studio session in a focused refinement pass.\n\n"
            f"BRIEF CHALLENGE: {brief['challenge']}\n\n"
            f"SPECIFIC RESEARCH QUESTION:\n{research_question}\n\n"
            f"Answer this one question with a named source, finding, and direct "
            f"relevance to the restated brief. Be concise — this is a targeted "
            f"response, not a full research pass. The session already has substantial "
            f"research context. Address only this specific gap."
        )

        focused_research = call_role(
            role="researcher",
            user_message=focused_research_task,
            context=brief["challenge"],
            blackboard=blackboard,
            model=SESSION_MODEL,
            max_tokens=1200,
            trait_profile=trait_profiles["researcher"],
            brief_doc=read_brief_doc(session_id),
            provider=PRIMARY_PROVIDER,
        )

        blackboard["research_trace"].append(
            {
                "research_id": f"R{len(blackboard['research_trace']) + 1:03d}",
                "initiated_by": "director_focused_refinement",
                "query": research_question,
                "sources": ["researcher_knowledge"],
                "summary": focused_research,
                "influence_on_ideation": "Focused refinement research delivered",
            }
        )
        update_brief_doc(
            session_id, "RESEARCHER", "RESEARCH — FOCUSED REFINEMENT", focused_research
        )
        scribe_log(
            blackboard,
            "RESEARCHER",
            "second_loop_research_delivered",
            f"Focused refinement research delivered. "
            f"Entry R{len(blackboard['research_trace']):03d} logged.",
        )
        print(focused_research)
        print()
    else:
        print("── RESEARCHER: SKIPPED ──────────────────────────────────────")
        print("  Existing research sufficient — no new call needed.")
        print()

    # ── Step 3: Creator focused pass ─────────────────────────────────────────
    print("── CREATOR: FOCUSED PASS ────────────────────────────────────")

    # Build full context for Creator — includes existing research
    existing_research_summary = "\n\n".join(
        r["summary"][:500] for r in blackboard["research_trace"]
    )

    creator_task_2 = (
        f"You are the Creator. The PIL has reacted to the first set of directions "
        f"and a focused refinement is needed.\n\n"
        f"ORIGINAL BRIEF: {brief['challenge']}\n\n"
        f"WHAT NEEDS TO CHANGE (from Director restatement):\n{restated_brief}\n\n"
        f"PIL REACTION:\n{all_signals}\n\n"
        f"EXISTING RESEARCH CONTEXT (summarized):\n{existing_research_summary}\n\n"
        + (f"NEW RESEARCH:\n{focused_research}\n\n" if focused_research else "")
        + f"Generate 1-2 focused directions (not three) that directly address "
        f"what the PIL identified as missing. Use Verbalized Sampling — assign "
        f"your bands after Step 0.\n\n"
        f"Step 0: Name what the first loop missed in one sentence. "
        f"Then generate from inside that gap.\n\n"
        f"For each direction:\n"
        f"IDEA_ID: [B1, B2]\n"
        f"BAND: [HIGH / MID / LOW]\n"
        f"PROBABILITY: [0.0–1.0]\n"
        f"TITLE: [memorable name]\n"
        f"CONCEPT: [2-3 sentences — what it is]\n"
        f"RATIONALE: [specifically how this addresses what the PIL said was missing]\n"
        f"EMOTIONAL TERRITORY: [how it feels]\n\n"
        f"These must address the gap the PIL identified — not variations on "
        f"what was already presented."
    )

    creator_output_2 = call_role(
        role="creator",
        user_message=creator_task_2,
        context=brief["challenge"],
        blackboard=blackboard,
        model=SESSION_MODEL,
        max_tokens=1800,
        trait_profile=trait_profiles["creator"],
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    blackboard["second_exploration"]["additional_cycles"].append(
        {
            "cycle_number": len(blackboard["ideation_cycles"]) + 1,
            "creator_proposals": [{"raw_response": creator_output_2}],
            "critic_feedback": [],
            "synthesized_directions": [],
        }
    )
    update_brief_doc(
        session_id, "CREATOR", "FOCUSED REFINEMENT DIRECTIONS", creator_output_2
    )
    scribe_log(
        blackboard,
        "CREATOR",
        "second_loop_ideation_complete",
        "Focused refinement: new directions generated targeting PIL gap.",
    )
    print(creator_output_2)
    print()

    # ── Step 4: Director selection ────────────────────────────────────────────
    # Build final candidate set based on rejection type:
    # flat    → new directions only (1-2)
    # partial → strongest survivor + new directions (max 3)
    # reframe → new directions only, framed as addressing the gap

    print("── DIRECTOR: SELECTION ──────────────────────────────────────")

    # Parse new B-series directions
    b_ids = re.findall(r"IDEA_ID:\s*(B\d+)", creator_output_2)
    b_titles = re.findall(r"TITLE:\s*(.+)", creator_output_2)

    new_candidates = []
    for i, bid in enumerate(b_ids[:2]):
        new_candidates.append(
            {
                "direction_id": f"C{i+1}",
                "internal_type": "second_loop",
                "description": b_titles[i].strip() if i < len(b_titles) else bid,
                "reasoning_summary": "",
                "research_influences": [],
                "critic_assessment": {
                    "strengths": [],
                    "concerns": [],
                    "refinement_notes": [],
                },
            }
        )

    # Build final candidate set
    if rejection_type == "partial" and surviving_direction:
        # Find surviving direction from first loop
        survivor = next(
            (
                c
                for c in blackboard["candidate_set"]
                if c["direction_id"] == surviving_direction
            ),
            None,
        )
        if survivor:
            # New directions first, survivor appended
            final_candidates = new_candidates + [survivor]
        else:
            final_candidates = new_candidates
    else:
        # flat or reframe: new directions only
        final_candidates = new_candidates

    # Renumber cleanly
    for i, c in enumerate(final_candidates):
        c["direction_id"] = f"C{i+1}"

    blackboard["candidate_set"] = final_candidates[:3]  # hard cap at 3

    # Update creator_output for Cell 11 synthesis
    creator_output = creator_output_2

    scribe_log(
        blackboard,
        "DIRECTOR",
        "second_loop_complete",
        f"Focused refinement complete. {len(blackboard['candidate_set'])} "
        f"candidates selected ({rejection_type} rejection). Presenting to PIL.",
    )

    print(f"  Rejection type : {rejection_type}")
    print(f"  Final set      : {len(blackboard['candidate_set'])} directions")
    for c in blackboard["candidate_set"]:
        print(
            f"    {c['direction_id']} [{c['internal_type']}]: {c['description'][:60]}"
        )
    print()

    # ── Step 5: Director presents refined directions to PIL ───────────────────
    print("── DIRECTOR: REFINED PRESENTATION ──────────────────────────")

    refined_set_summary = "\n".join(
        f"- {c['direction_id']}: {c['description']}"
        for c in blackboard["candidate_set"]
    )

    # Frame the presentation based on rejection type
    if rejection_type == "flat":
        framing_note = (
            "The PIL rejected all first directions. Present these as a completely "
            "fresh approach that addresses what they said was missing. Do not "
            "reference the first directions at all."
        )
    elif rejection_type == "partial":
        framing_note = (
            f"The PIL showed some energy toward {surviving_direction or 'one direction'} "
            f"but it didn't fully land. Present the new directions as addressing the gap, "
            f"and carry the strongest first-loop direction forward as still relevant."
        )
    else:  # reframe
        framing_note = (
            "The PIL identified a missing dimension. Present these as directly "
            "addressing that dimension — not as replacements for everything, "
            "but as what was missing from the first pass."
        )

    presentation_task_2 = (
        f"Present these refined directions to the PIL.\n\n"
        f"BRIEF: {brief['challenge']}\n\n"
        f"WHAT PIL SAID WAS MISSING: {restated_brief}\n\n"
        f"DIRECTIONS TO PRESENT:\n{refined_set_summary}\n\n"
        f"FULL CREATOR OUTPUT FOR REFERENCE:\n{creator_output_2}\n\n"
        f"FRAMING GUIDANCE: {framing_note}\n\n"
        f"Present each direction clearly with 2-3 sentences of substance. "
        f"Include one transparency hook per direction — a brief sentence "
        f"inviting the PIL to ask about research or evaluation if they want "
        f"to go deeper. "
        f"Close by asking which direction resonates, or whether they want to "
        f"proceed directly to synthesis."
    )

    refined_presentation = call_role(
        role="director",
        user_message=presentation_task_2,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_MODEL,
        max_tokens=600,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    print(refined_presentation)
    print()

    # ── Step 6: PIL reacts to refined directions ──────────────────────────────
    pil_refined_reaction = input("Your reaction: ").strip()

    # ── Step 7: Director reads reaction and logs ──────────────────────────────
    refined_signal_task = (
        f"The PIL has responded to the refined directions:\n\n"
        f"REFINED DIRECTIONS PRESENTED:\n{refined_set_summary}\n\n"
        f"PIL RESPONSE:\n{pil_refined_reaction}\n\n"
        f"Read this response. In one or two sentences:\n"
        f"1. Acknowledge what they said naturally (do not be sycophantic)\n"
        f"2. Confirm which direction or combination you will synthesize toward\n\n"
        f"Keep it brief and decisive. The PIL has given their input — "
        f"now confirm the direction and move toward synthesis."
    )

    refined_signal_response = call_role(
        role="director",
        user_message=refined_signal_task,
        context=brief["challenge"],
        blackboard=blackboard,
        model=DIRECTOR_FAST_MODEL,
        max_tokens=150,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )

    print()
    print(refined_signal_response)
    print()

    # Store refined reaction signals
    blackboard["user_response"]["signals_extracted"].append(
        f"REFINED PASS SIGNAL: {pil_refined_reaction}"
    )
    blackboard["second_exploration"][
        "reason"
    ] += f"\n\nRefined pass PIL reaction: {pil_refined_reaction}"

    update_brief_doc(
        session_id,
        "DIRECTOR",
        "REFINED PRESENTATION + PIL REACTION",
        f"**Director presented:**\n{refined_presentation}\n\n"
        f"**PIL responded:**\n{pil_refined_reaction}\n\n"
        f"**Director acknowledged:**\n{refined_signal_response}",
    )
    scribe_log(
        blackboard,
        "DIRECTOR",
        "refined_presentation_complete",
        f"Refined directions presented and PIL reacted. "
        f"Signal: '{pil_refined_reaction[:60]}'",
    )

    print("── FOCUSED REFINEMENT COMPLETE ──────────────────────────────")
    print(f"  Final candidates  : {len(blackboard['candidate_set'])}")
    for c in blackboard["candidate_set"]:
        print(
            f"    {c['direction_id']} [{c['internal_type']}]: {c['description'][:60]}"
        )
    print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 8c — FOCUSED REFINEMENT
════════════════════════════════════════════════════════════
✓ Second loop not required — proceeding to synthesis
✓ Reasoning trace: 41 entries


---
## Cell 11 — Stage 9: Final Synthesis

Director produces the final synthesis — refining the chosen direction
and incorporating signals from the PIL's reaction.
This is the studio's best work.

In [21]:
# ── STAGE 9: FINAL SYNTHESIS ─────────────────────────────────────────────────
# v3.2 patch: research-anchored action plan, max_tokens raised to 1400

print("═" * 60)
print("STAGE 9 — FINAL SYNTHESIS")
print("═" * 60)

all_signals = "\n\n".join(blackboard["user_response"]["signals_extracted"])
signals = all_signals if all_signals else ""

# Pull research summary from research_trace for synthesis integration
research_summary = ""
if blackboard.get("research_trace"):
    research_summary = "\n\n".join(
        f"FINDING: {entry.get('summary', '')[:600]}"
        for entry in blackboard["research_trace"]
        if entry.get("summary")
    )

synthesis_task = f"""Produce the final synthesis for the PIL.

BRIEF: {brief['challenge']}
DESIRED RESULT: {brief['desired_result']}

PIL REACTION: {pil_reaction}

SIGNALS EXTRACTED:
{signals}

STUDIO DIRECTIONS EXPLORED:
{creator_output[:400]}

RESEARCH PRODUCED BY THE STUDIO:
{research_summary[:2000]}

Synthesize the most compelling direction — refined by the PIL's signals.
This may combine elements from multiple directions.

Name this document appropriately for the type of problem solved. Use terms
such as: Action Plan, Creative Direction, Business Launch Plan, Brand Strategy,
Product Roadmap, or similar. Do not use "Final Synthesis" under any circumstances.

Begin with a single sentence introduction — warm, direct, reflective of genuine
authorship by the full creative team, informed by everything the PIL shared.

Structure:

[DOCUMENT NAME]: [title]

[intro sentence]

CORE CONCEPT:
3-4 sentences. The essential idea, grounded in what discovery and research revealed.

WHY THIS WORKS:
2-3 sentences. Name the specific insight from the research or evaluation that makes
this direction sound — not just logical, but evidenced. If the research found a
named precedent, a documented outcome, or a specific mechanism that validates this
direction, name it here in plain language.

ACTION PLAN:
Organize as 2-3 phases, not a flat list. Each phase has:
- A short phase name and approximate time horizon
- 1-2 specific actions
- One sentence connecting the action to what the research found — why this
  specific approach is the right move, drawn from the studio's findings

The research connection should be concrete and specific, not generic. If the
research named a methodology, a case study, a mechanism of failure, or a
documented outcome that directly informs an action step, use it. This is what
makes the plan feel earned rather than invented.

Do not fabricate contact details, URLs, or current organizational information —
these require live verification. Where specific people or organizations are named,
note them as starting points to research rather than confirmed contacts.

Close with a single sentence inviting the PIL's reaction. Open and genuine.

Tone: confident, clear, earned. The PIL should leave feeling that the studio
understood their problem and built something real for them."""

synthesis_response = call_role(
    role="director",
    user_message=synthesis_task,
    context=brief["challenge"],
    blackboard=blackboard,
    model=DIRECTOR_MODEL,
    max_tokens=1400,
    brief_doc=read_brief_doc(session_id),
    provider=PRIMARY_PROVIDER,
)

# Store final synthesis
synthesis = blackboard["final_synthesis"]
synthesis["summary"] = synthesis_response
if blackboard["candidate_set"]:
    synthesis["final_direction"] = blackboard["candidate_set"][-1]["direction_id"]
synthesis["refinements"].append(f"Refined from PIL signal: '{pil_reaction[:100]}'")

scribe_log(
    blackboard,
    "DIRECTOR",
    "synthesis_complete",
    "Final synthesis delivered to PIL. Session complete.",
)

print("── FINAL SYNTHESIS ─────────────────────────────────────────")
print(synthesis_response)
print("────────────────────────────────────────────────────────────")
print(f"\n✓ Final synthesis complete")
print(f"✓ Reasoning trace: {len(blackboard['reasoning_trace'])} entries")

════════════════════════════════════════════════════════════
STAGE 9 — FINAL SYNTHESIS
════════════════════════════════════════════════════════════
── FINAL SYNTHESIS ─────────────────────────────────────────
Decision Framework: Choose the place that can support your actual way of living, not the one that only wins in theory.

You do not need another abstract comparison here — you need a way to identify which option stays livable after the first rush of conviction wears off.

CORE CONCEPT:
The strongest path is not to pick between two imperfect locations at face value, but to evaluate them in sequence: first by the practices your life actually depends on, then by whether each place can host those practices without draining you, and only then by whether the surviving option can be made financially workable without hollowing out what made it attractive. That sequencing matters because cost is not the first question if a place cannot support the life you are trying to live. And emotion is

---
## Cell 11b — Stage 9b: Research  Appendix

The studio's full research output, surfaced after synthesis for PIL review and follow-up.

In [22]:
# ── STAGE 9b: RESEARCH APPENDIX ──────────────────────────────────────────────
# Prints full research trace as a readable appendix after final synthesis.
# All findings surfaced for PIL review. Future: downloadable document.

print("═" * 60)
print("RESEARCH APPENDIX")
print("Studio findings that informed this session")
print("═" * 60)

research_trace = blackboard.get("research_trace", [])

if not research_trace:
    print("No research findings recorded in this session.")
else:
    print(f"This session produced {len(research_trace)} research contribution(s).\n")
    print(
        "These findings informed the directions and synthesis above.\n"
        "They are presented here in full for your reference.\n"
    )
    print("─" * 60)

    for i, entry in enumerate(research_trace):
        summary = entry.get("summary", "")
        if not summary:
            continue

        print(f"\nRESEARCH CONTRIBUTION {i + 1}")
        print(f"Initiated by: {entry.get('initiated_by', 'creative_team')}")
        print(f"Query: {entry.get('query', '')}")
        print("─" * 40)
        print(summary)
        print()

    print("─" * 60)
    print(
        "\nNote: Research findings are drawn from the studio's training knowledge.\n"
        "Named sources, individuals, and institutions should be independently\n"
        "verified before being used in outreach or decision-making.\n"
        "Links and current contact information require live web research."
    )

print(f"\n✓ Research appendix complete")
print(f"✓ {len(research_trace)} research entry/entries surfaced")

════════════════════════════════════════════════════════════
RESEARCH APPENDIX
Studio findings that informed this session
════════════════════════════════════════════════════════════
This session produced 1 research contribution(s).

These findings informed the directions and synthesis above.
They are presented here in full for your reference.

────────────────────────────────────────────────────────────

RESEARCH CONTRIBUTION 1
Initiated by: creative_team
Query: Targeted requests + autonomous contribution post-ideation
────────────────────────────────────────
CITED FINDING  
TOPIC: Reducing the cost of a “dream” move while preserving place-feel  
SOURCE: Mihaly Csikszentmihalyi and Rochberg-Halton, *The Meaning of Things* (1981)  
FINDING: The book distinguishes between possessions and environments that support identity, autonomy, and daily meaning versus those that merely satisfy practical need. In their interviews, people often described meaningful places in terms of continuity, con

---
## Cell 12 — Session Record: Blackboard Inspection + Save

Complete session record. Full reasoning trace visible.
Session saved to JSON — raw material for Phase 3 visualization.

In [23]:
# ── SESSION RECORD ────────────────────────────────────────────────────────────

print("═" * 60)
print("SESSION RECORD")
print("═" * 60)
print(f"Session ID      : {blackboard['session_metadata']['session_id']}")
print(f"Prompt          : {blackboard['user_prompt']}")
print(f"Model           : {SESSION_MODEL}")
print()
print(f"Ideation cycles : {len(blackboard['ideation_cycles'])}")
print(f"Research entries: {len(blackboard['research_trace'])}")
print(f"Candidate dirs  : {len(blackboard['candidate_set'])}")
print(f"Reasoning steps : {len(blackboard['reasoning_trace'])}")
print()
print("── REASONING TRACE ─────────────────────────────────────────")
for entry in blackboard["reasoning_trace"]:
    print(
        f"  {entry['step']:>2} | {entry['role']:<12} | {entry['action']:<22} | {entry['summary'][:60]}"
    )

print()
print("── CANDIDATE DIRECTIONS ────────────────────────────────────")
for c in blackboard["candidate_set"]:
    print(f"  {c['direction_id']} | {c['internal_type']:<22} | {c['description']}")

# -- Director summary ----------------------------------------------------------
# One-sentence session label written by the Director.
# Stored in metadata for quick identification without opening the full JSON.
# Fix: pass read_brief_doc(session_id) so the Director has full session context.
print()
print("── DIRECTOR SUMMARY ────────────────────────────────────────")

summary_task = (
    "In one sentence, summarize what problem was explored in this session "
    "and what direction the studio landed on. "
    "Write it as a neutral third-party description — not addressed to anyone, "
    "not promotional, not a question. "
    "Be specific to this session: name the domain, the core challenge, and "
    "the nature of the solution direction. Maximum 35 words."
)

try:
    director_summary = call_role(
        role="director",
        user_message=summary_task,
        context=blackboard["user_prompt"],
        blackboard=blackboard,
        model=DIRECTOR_FAST_MODEL,
        max_tokens=80,
        brief_doc=read_brief_doc(session_id),
        provider=PRIMARY_PROVIDER,
    )
    director_summary = director_summary.strip()
    blackboard["session_metadata"]["director_summary"] = director_summary
    print(f"  {director_summary}")
except Exception as e:
    print(f"  ⚠ Director summary failed: {e}")
    blackboard["session_metadata"]["director_summary"] = ""

# -- Assemble evaluation payload -----------------------------------------------
# Packages problem_prompt, baseline, and final synthesis for evaluator.
assemble_evaluation_payload(blackboard, baseline_response)

# -- Post-session notes --------------------------------------------------------
# Capture observer notes while session is fresh. Stored in JSON metadata.
session_notes = input("\nSession notes (press Enter to skip):\n> ").strip()
if session_notes:
    blackboard["session_metadata"]["notes"] = session_notes

# -- Save full session ---------------------------------------------------------
saved_path = save_session(blackboard)
print()
print(f"✓ Full session saved to: {saved_path}")
print()
print("Phase 1 complete. Full studio workflow executed.")
print()
print("This JSON is Phase 3 input — the semantic trajectory data.")
print("Every reasoning_trace entry is an utterance in conceptual space.")

════════════════════════════════════════════════════════════
SESSION RECORD
════════════════════════════════════════════════════════════
Session ID      : 321d6ab4-4c48-4317-be5b-b01036e06baa
Prompt          : I have two places I could move to—one feels right but makes no financial sense, the other is practical but I don't feel anything—how do I decide?
Model           : gpt-5.4-mini

Ideation cycles : 2
Research entries: 1
Candidate dirs  : 3
Reasoning steps : 43

── REASONING TRACE ─────────────────────────────────────────
   1 | SYSTEM       | session_start          | Session initiated: 'I have two places I could move to—one fe
   2 | DIRECTOR     | api_call               | [openai] Responded to: 'A person has just arrived at The Cre
   3 | DIRECTOR     | routing_studio         | Request routed to STUDIO. Entering adaptive discovery.
   4 | DIRECTOR     | api_call               | [openai] Responded to: 'A person has arrived at The Creative
   5 | DIRECTOR     | discovery_opened     